# 1. Loading Standardized Sensor Datasets

### Objective

Load the eight standardized sensor datasets produced during the previous stage.

### Why?

The standardized datasets provide a consistent structure across participants while preserving original measurements, missing values, duplicate timestamps and validity flags.

We will use these datasets as inputs for cleaning. The original standardized files will remain unchanged throughout the process.

### Cleaning principle

All cleaning operations will be performed on separate working copies. No records will be permanently removed or modified without an explicitly documented cleaning rule.

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

# Locate project directory
project_path = Path.cwd()

if project_path.name == "notebooks":
    project_path = project_path.parent

# Input and output directories
input_path = (
    project_path / "data" / "interim" /
    "standardized" / "sensors"
)

output_path = (
    project_path / "data" / "processed" / "cleaned"
)

output_path.mkdir(parents=True, exist_ok=True)

# Sensor names
sensors = [
    "BRI", "EI", "MI", "PEI",
    "PRVI", "SHRI", "SII", "SQI"
]

# Load standardized datasets
sensor_data = {}

for sensor in sensors:

    file_path = input_path / f"{sensor}.parquet"

    sensor_data[sensor] = pd.read_parquet(file_path)

    print(
        f"{sensor}: "
        f"{sensor_data[sensor].shape[0]} rows, "
        f"{sensor_data[sensor].shape[1]} columns"
    )

# Create separate working copies
sensor_working = {
    sensor: df.copy(deep=True)
    for sensor, df in sensor_data.items()
}

print("\nAll standardized datasets loaded.")
print("Separate working copies created.")


## 2. Inspecting Sensor Columns and Data Types

**Objective:** Understand the structure and contents of each standardized sensor dataset before applying cleaning operations.

For each sensor, we inspected:
- Available columns and their meanings.
- Data types of individual columns.
- Number of missing values in each column.
- A sample of the first three records.

**Why this step is necessary:**

Different sensors contain different measurements and may have different missing-data patterns. Understanding these differences helps us establish appropriate sensor-specific cleaning rules rather than applying one generic cleaning procedure.

**Important:** No values have been modified, deleted, imputed, or filtered during this inspection.

In [ ]:
for sensor, df in sensor_working.items():
    print(f"\n{'=' * 60}")
    print(f"SENSOR: {sensor}")
    print(f"{'=' * 60}")

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    if missing.empty:
        print("No missing values.")
    else:
        print(missing)

    print("\nFirst 3 rows:")
    display(df.head(3))

## 3. Examining Validity Flags and Zero Measurements

**Objective:** Investigate the relationship between sensor measurements, zero-valued records, missing measurements, and the two validity flags.

For each sensor, we calculated:

- Number of rows where all available measurement values are zero.
- Number of rows where all measurement columns are missing.
- Frequency of each `data_validity` and `index_validity` combination, including missing flag values.

**Why this step is necessary:**

Zero-valued measurements should not automatically be treated as missing data. They may represent invalid measurements, unavailable sensor readings, or genuine recorded values. The validity flags provide additional context for interpreting these records.

**Important distinction:**
- `data_validity` concerns the validity of the underlying sensor data.
- `index_validity` concerns the validity of the calculated health index.

These flags must be examined separately before deciding whether any measurements or records should be excluded from subsequent analysis.

**No rows or values were modified during this investigation.**

In [ ]:
for sensor, df in sensor_working.items():

    metadata_cols = [
        "Timestamp",
        "Participant_ID",
        "Country",
        "Sensor",
        "index_validity",
        "data_validity"
    ]

    measurement_cols = [
        col for col in df.select_dtypes(include="number").columns
        if col not in metadata_cols
    ]

    measurements = df[measurement_cols]

    # Rows where every available measurement is zero.
    # Missing values are ignored for this particular check.
    has_measurement = measurements.notna().any(axis=1)

    all_observed_zero = (
        has_measurement &
        measurements.fillna(0).eq(0).all(axis=1)
    )

    # Rows where every measurement is missing.
    all_measurements_missing = measurements.isna().all(axis=1)

    print(f"\n{'=' * 60}")
    print(f"SENSOR: {sensor}")
    print(f"{'=' * 60}")

    print(f"Total rows: {len(df)}")
    print(f"Measurement columns: {len(measurement_cols)}")

    print(
        "Rows where all observed measurements are zero:",
        int(all_observed_zero.sum())
    )

    print(
        "Rows where all measurements are missing:",
        int(all_measurements_missing.sum())
    )

    print("\nValidity flag combinations:")
    flag_counts = (
        df.groupby(
            ["data_validity", "index_validity"],
            dropna=False
        )
        .size()
        .rename("row_count")
        .reset_index()
    )

    display(flag_counts)

## 3.1 Initial Findings: Validity Flags and Zero Measurements

### Observed results

The analysis identified substantial numbers of all-zero measurement rows across most sensor streams.

- BRI: 9,166 all-zero rows.
- EI: 15,017 all-zero rows.
- MI: 6,195 all-zero rows.
- PEI: 7,304 all-zero rows.
- PRVI: 0 all-zero rows and 505 rows with all measurement columns missing.
- SHRI: 9,324 all-zero rows.
- SII: 14,566 all-zero rows.
- SQI: 9,359 all-zero rows.

The validity flag combinations also differ across sensors. In particular, MI contains 3,370 records with valid underlying data but an invalid calculated index.

### Interpretation

1. Zero-valued measurements cannot automatically be considered missing or invalid.
2. `data_validity` and `index_validity` represent different validity concepts and must be handled separately.
3. PRVI has a distinct missing-data pattern, including 505 rows with missing measurement values and missing `index_validity`.
4. Records with valid underlying data but invalid calculated indices may still contain useful measurements.

### Decision

No records will be deleted, and no missing or zero values will be replaced at this stage. We will investigate the relationship between all-zero measurements and validity flags before establishing sensor-specific cleaning rules.

## 4. Cross-Checking Zero Measurements Against Validity Flags

**Objective:** Determine how all-zero and all-missing measurement rows are distributed across the validity flag combinations.

For each sensor, we will calculate:
- Total records in each validity flag combination.
- Number of all-zero measurement records in each combination.
- Number of all-missing measurement records in each combination.
- Number of remaining records that contain at least some non-zero measurement information.

**Why this step is necessary:**

The previous analysis calculated zero-measurement counts and validity flag counts separately. Comparing these counts directly does not establish whether zero-valued measurements occur in valid or invalid records.

This cross-check will help us identify which records may be unsuitable for analysis and which measurements should be retained.

No records will be modified or deleted.

In [ ]:
for sensor, df in sensor_working.items():

    metadata_cols = [
        "Timestamp",
        "Participant_ID",
        "Country",
        "Sensor",
        "index_validity",
        "data_validity"
    ]

    measurement_cols = [
        col for col in df.select_dtypes(include="number").columns
        if col not in metadata_cols
    ]

    measurements = df[measurement_cols]

    has_measurement = measurements.notna().any(axis=1)

    all_zero = (
        has_measurement &
        measurements.fillna(0).eq(0).all(axis=1)
    )

    all_missing = measurements.isna().all(axis=1)

    audit = df[
        ["data_validity", "index_validity"]
    ].copy()

    audit["all_zero"] = all_zero
    audit["all_missing"] = all_missing

    summary = (
        audit.groupby(
            ["data_validity", "index_validity"],
            dropna=False
        )
        .agg(
            total_rows=("all_zero", "size"),
            all_zero_rows=("all_zero", "sum"),
            all_missing_rows=("all_missing", "sum")
        )
        .reset_index()
    )

    summary["other_rows"] = (
        summary["total_rows"]
        - summary["all_zero_rows"]
        - summary["all_missing_rows"]
    )

    print(f"\n{'=' * 60}")
    print(f"SENSOR: {sensor}")
    print(f"{'=' * 60}")

    display(summary)

## 4.1 Findings: Zero Measurements and Validity Flags

### Observed results

The cross-check established the following:

- **BRI:** All 9,166 all-zero rows have `index_validity = 0`.
- **EI:** 13 all-zero rows have both validity flags equal to 1.
- **MI:** 222 rows have non-zero measurements, `data_validity = 1`, and `index_validity = 0`.
- **PEI:** All 7,304 all-zero rows have both validity flags equal to 0.
- **PRVI:** 505 rows have all measurement columns missing and a missing `index_validity`; the other 2,412 rows have both flags equal to 1.
- **SHRI:** All 9,324 all-zero rows have `index_validity = 0`.
- **SII:** 10 all-zero rows have both validity flags equal to 1.
- **SQI:** All 9,359 all-zero rows have `index_validity = 0`.

### Interpretation

The relationship between zero measurements and validity flags is not identical across all sensors.

In particular:
- The 13 EI and 10 SII exceptions require investigation because they contain all-zero measurements despite both flags indicating validity.
- The 222 MI records show that an invalid calculated index does not necessarily mean that all underlying measurements are zero or unavailable.
- PRVI has a separate missing-data pattern and should not be handled using the same zero-value rules.

### Preliminary cleaning decision

No rows will be deleted based solely on zero measurements or `index_validity`.

Both validity flags will be preserved. Sensor-specific handling will be determined after investigating the exceptional records.

## 5. Investigating Exceptional Sensor Records

**Objective:** Investigate unusual records identified during the validity-flag and zero-measurement analysis.

Three specific cases are examined:

1. EI records containing all-zero measurements despite both validity flags being valid.
2. SII records containing all-zero measurements despite both validity flags being valid.
3. MI records containing non-zero measurements and valid underlying data, but an invalid calculated index.

**Investigation approach:**

- Extract the exceptional EI and SII records.
- Identify which participants and timestamps are associated with these records.
- Examine the corresponding measurement values.
- For MI, examine the distribution of exceptional records across participants and calculate descriptive statistics for their measurements.

**Why this matters:**

These records do not follow the dominant patterns observed in their respective sensor streams. Investigating them helps us distinguish potentially unusual sensor behavior from genuine measurement patterns or data-quality issues.

**Important:** These records are being investigated, not automatically classified as errors. No records or measurements are modified or deleted.

In [ ]:
# Investigate EI and SII: all-zero measurements despite both flags being 1

for sensor in ["EI", "SII"]:

    df = sensor_working[sensor]

    metadata_cols = [
        "Timestamp",
        "Participant_ID",
        "Country",
        "Sensor",
        "index_validity",
        "data_validity"
    ]

    measurement_cols = [
        col for col in df.select_dtypes(include="number").columns
        if col not in metadata_cols
    ]

    all_zero = (
        df[measurement_cols].notna().any(axis=1) &
        df[measurement_cols].fillna(0).eq(0).all(axis=1)
    )

    exceptions = df[
        all_zero &
        (df["data_validity"] == 1) &
        (df["index_validity"] == 1)
    ]

    print(f"\n{'=' * 60}")
    print(f"{sensor}: ALL-ZERO RECORDS WITH BOTH FLAGS = 1")
    print(f"Number of records: {len(exceptions)}")
    display(exceptions)


# Investigate MI: non-zero measurements with valid data but invalid index

df = sensor_working["MI"]

metadata_cols = [
    "Timestamp",
    "Participant_ID",
    "Country",
    "Sensor",
    "index_validity",
    "data_validity"
]

measurement_cols = [
    col for col in df.select_dtypes(include="number").columns
    if col not in metadata_cols
]

all_zero = (
    df[measurement_cols].notna().any(axis=1) &
    df[measurement_cols].fillna(0).eq(0).all(axis=1)
)

mi_exceptions = df[
    (df["data_validity"] == 1) &
    (df["index_validity"] == 0) &
    (~all_zero)
]

print(f"\n{'=' * 60}")
print("MI: NON-ZERO RECORDS WITH INVALID INDEX")
print(f"Number of records: {len(mi_exceptions)}")

print("\nRecords per participant:")
display(
    mi_exceptions.groupby("Participant_ID")
    .size()
    .sort_values(ascending=False)
)

print("\nMeasurement statistics:")
display(mi_exceptions[measurement_cols].describe().T)

print("\nFirst 20 exceptional records:")
display(mi_exceptions.head(20))

## 6. Investigating Time Alignment of EI and SII Exceptions

**Objective:** Determine whether all-zero EI and SII records with both validity flags equal to 1 occur at matching participant timestamps, and inspect the availability of other sensor observations around those times.

**Approach:**

1. Extract all-zero EI and SII records with both validity flags equal to 1.
2. Compare their participant and timestamp combinations.
3. Identify matching exceptions across the two streams.
4. Check whether other sensor streams contain observations for the same participants within 24 hours of each matching exception.

**Why this matters:**

If multiple sensor streams show exceptional records at matching times, this may indicate a shared data-collection or recording issue. If other streams contain normal observations, the issue may be more specific to EI and SII.

**Important:** Temporal proximity does not prove that measurements were collected simultaneously or that they share a common cause. This is an exploratory data-quality investigation. No records are modified or deleted.

In [ ]:
# Step 1: Extract exceptional EI and SII records

def get_all_zero_valid_records(sensor):

    df = sensor_working[sensor]

    metadata_cols = [
        "Timestamp", "Participant_ID", "Country",
        "Sensor", "index_validity", "data_validity"
    ]

    measurement_cols = [
        col for col in df.select_dtypes(include="number").columns
        if col not in metadata_cols
    ]

    all_zero = (
        df[measurement_cols].notna().any(axis=1) &
        df[measurement_cols].fillna(0).eq(0).all(axis=1)
    )

    return df[
        all_zero &
        (df["data_validity"] == 1) &
        (df["index_validity"] == 1)
    ].copy()


ei_exceptions = get_all_zero_valid_records("EI")
sii_exceptions = get_all_zero_valid_records("SII")

# Step 2: Compare participant and timestamp combinations

ei_keys = set(zip(
    ei_exceptions["Participant_ID"],
    ei_exceptions["Timestamp"]
))

sii_keys = set(zip(
    sii_exceptions["Participant_ID"],
    sii_exceptions["Timestamp"]
))

matching_keys = ei_keys.intersection(sii_keys)

print("EI exceptions:", len(ei_keys))
print("SII exceptions:", len(sii_keys))
print("Matching participant-timestamp pairs:", len(matching_keys))

# Step 3: Display matching exceptions

matching_records = pd.DataFrame(
    sorted(matching_keys),
    columns=["Participant_ID", "Timestamp"]
)

display(matching_records)


# Step 4: Check other sensor streams within +/- 24 hours

other_sensors = ["BRI", "MI", "PEI", "SHRI", "SQI"]

for sensor in other_sensors:

    df = sensor_working[sensor]

    matches = []

    for _, exception in matching_records.iterrows():

        participant = exception["Participant_ID"]
        timestamp = exception["Timestamp"]

        nearby = df[
            (df["Participant_ID"] == participant) &
            (df["Timestamp"] >= timestamp - pd.Timedelta(hours=24)) &
            (df["Timestamp"] <= timestamp + pd.Timedelta(hours=24))
        ]

        if not nearby.empty:

            matches.append({
                "Participant_ID": participant,
                "Exception_Timestamp": timestamp,
                "Nearby_Observations": len(nearby),
                "First_Observation": nearby["Timestamp"].min(),
                "Last_Observation": nearby["Timestamp"].max()
            })

    print(f"\n{sensor}: observations within +/- 24 hours")
    display(pd.DataFrame(matches))

## 6. Findings: Time Alignment of EI and SII Exceptions

### Observed results

- EI contained 13 records with all measurement values equal to zero despite `data_validity = 1` and `index_validity = 1`.
- SII contained 10 records with the same pattern.
- All 10 SII exceptions matched EI exceptions at the exact same participant and timestamp.
- Eight of these matching timestamps belonged to participant SPA-015.
- The three additional EI exceptions did not have matching SII exceptions.
- For all 10 matching timestamps, BRI, MI, PEI, SHRI, and SQI each had at least one observation within ±24 hours.

### Interpretation

The repeated EI/SII pattern suggests that these exceptions may be related to a measurement-specific or index-calculation issue rather than a complete interruption of monitoring. Other sensor streams continued to have nearby observations.

However, the ±24-hour comparison only establishes temporal proximity. It does not prove that other sensors recorded valid measurements at the exact same time or establish the cause of the EI/SII exceptions.

### Preprocessing decision

- Retain all exceptional records.
- Do not replace their zero values or remove them at this stage.
- Preserve the original validity flags.
- Treat these records as suspicious zero-value observations for later quality-aware feature engineering.

**Important:** No source data was modified during this investigation.

## 7. Investigating Duplicate Participant-Timestamp Records

### Objective

Identify records that share the same `Participant_ID` and `Timestamp` in EI, SII, and PRVI.

For each duplicate group, compare the measurement columns and validity flags to determine whether the records are identical or contain conflicting information.

### Approach

- Group records by participant and timestamp.
- Count duplicate groups and additional rows.
- Check whether measurement values differ within each group.
- Check whether `index_validity` or `data_validity` differs.
- Display a small sample of conflicting groups for manual inspection.

No duplicate records will be removed or modified during this investigation.

In [ ]:
# Step 1: Define the sensors and columns used for grouping

duplicate_sensors = ["EI", "SII", "PRVI"]

key_cols = ["Participant_ID", "Timestamp"]
flag_cols = ["index_validity", "data_validity"]


# Step 2: Investigate duplicates in each sensor

for sensor in duplicate_sensors:

    df = sensor_working[sensor]

    # Identify all rows belonging to duplicate groups
    duplicate_rows = df[
        df.duplicated(subset=key_cols, keep=False)
    ].copy()

    # Group duplicate rows by participant and timestamp
    duplicate_groups = list(
        duplicate_rows.groupby(key_cols, dropna=False)
    )

    measurement_cols = [
        col for col in df.columns
        if col not in key_cols + [
            "Country", "Sensor"
        ] + flag_cols
    ]

    measurement_conflicts = []
    flag_conflicts = []

    for group_key, group in duplicate_groups:

        # Check whether any measurement column has different values
        differing_measurements = [
            col for col in measurement_cols
            if group[col].nunique(dropna=False) > 1
        ]

        # Check whether validity flags differ
        differing_flags = [
            col for col in flag_cols
            if col in group.columns
            and group[col].nunique(dropna=False) > 1
        ]

        if differing_measurements:
            measurement_conflicts.append({
                "Participant_ID": group_key[0],
                "Timestamp": group_key[1],
                "Differing_Measurements": differing_measurements
            })

        if differing_flags:
            flag_conflicts.append({
                "Participant_ID": group_key[0],
                "Timestamp": group_key[1],
                "Differing_Flags": differing_flags
            })

    # Step 3: Print summary

    print(f"\n{'=' * 60}")
    print(f"SENSOR: {sensor}")
    print(f"{'=' * 60}")

    print("Duplicate groups:", len(duplicate_groups))
    print("Rows involved in duplicates:", len(duplicate_rows))
    print("Extra duplicate rows:", len(duplicate_rows) - len(duplicate_groups))

    print("Groups with measurement conflicts:", len(measurement_conflicts))
    print("Groups with validity flag conflicts:", len(flag_conflicts))

    # Step 4: Display conflict summaries

    print("\nMeasurement conflict examples:")
    display(pd.DataFrame(measurement_conflicts).head(5))

    print("\nValidity flag conflict examples:")
    display(pd.DataFrame(flag_conflicts).head(5))

    # Step 5: Display the actual rows for the first conflicting groups

    conflicting_keys = {
        (item["Participant_ID"], item["Timestamp"])
        for item in measurement_conflicts + flag_conflicts
    }

    print("\nActual duplicate records from the first 3 conflicting groups:")

    for participant, timestamp in list(sorted(conflicting_keys))[:3]:

        example = duplicate_rows[
            (duplicate_rows["Participant_ID"] == participant) &
            (duplicate_rows["Timestamp"] == timestamp)
        ]

        display(example)

## 8. Detailed Analysis of Duplicate Conflicts

### Objective

Examine the nature and extent of duplicate-record conflicts in EI, SII, and PRVI.

For EI and SII, distinguish completely identical duplicate groups from groups with conflicting validity flags.

For PRVI, identify which measurement columns differ, determine whether differences are caused by missing values or numerical disagreements, and quantify the numerical differences where both records contain values.

### Important consideration

Duplicate records will remain unchanged during this analysis. The purpose is to understand their structure before selecting an appropriate handling strategy.

In [ ]:
# Step 1: Define duplicate grouping and column categories

key_cols = ["Participant_ID", "Timestamp"]
flag_cols = ["index_validity", "data_validity"]

for sensor in ["EI", "SII", "PRVI"]:

    df = sensor_working[sensor]

    measurement_cols = [
        col for col in df.select_dtypes(include="number").columns
        if col not in flag_cols
    ]

    duplicate_rows = df[
        df.duplicated(subset=key_cols, keep=False)
    ]

    groups = duplicate_rows.groupby(key_cols, dropna=False)

    print(f"\n{'=' * 65}")
    print(f"{sensor}: DUPLICATE GROUP CLASSIFICATION")
    print(f"{'=' * 65}")

    completely_identical = 0
    flag_only_conflicts = 0
    measurement_conflicts = 0

    for _, group in groups:

        measurement_diff = any(
            group[col].nunique(dropna=False) > 1
            for col in measurement_cols
        )

        flag_diff = any(
            group[col].nunique(dropna=False) > 1
            for col in flag_cols
        )

        if measurement_diff:
            measurement_conflicts += 1
        elif flag_diff:
            flag_only_conflicts += 1
        else:
            completely_identical += 1

    print("Completely identical groups:", completely_identical)
    print("Groups differing only in validity flags:", flag_only_conflicts)
    print("Groups with measurement conflicts:", measurement_conflicts)


# Step 2: Identify which PRVI measurement columns conflict most frequently

df = sensor_working["PRVI"]

duplicate_rows = df[
    df.duplicated(subset=key_cols, keep=False)
]

prvi_groups = duplicate_rows.groupby(key_cols, dropna=False)

conflict_counts = {col: 0 for col in measurement_cols}

# Recalculate PRVI measurement columns explicitly
prvi_measurement_cols = [
    col for col in df.select_dtypes(include="number").columns
    if col not in flag_cols
]

conflict_counts = {col: 0 for col in prvi_measurement_cols}

for _, group in prvi_groups:

    for col in prvi_measurement_cols:

        if group[col].nunique(dropna=False) > 1:
            conflict_counts[col] += 1

print("\nPRVI measurement conflicts by column:")

display(
    pd.DataFrame(
        conflict_counts.items(),
        columns=["Measurement", "Conflicting_Groups"]
    ).sort_values("Conflicting_Groups", ascending=False)
)


# Step 3: Distinguish missing-value differences from numerical differences

missingness_conflicts = {col: 0 for col in prvi_measurement_cols}
numeric_conflicts = {col: 0 for col in prvi_measurement_cols}

for _, group in prvi_groups:

    for col in prvi_measurement_cols:

        if group[col].nunique(dropna=False) > 1:

            if group[col].isna().any():
                missingness_conflicts[col] += 1

            if group[col].notna().all():
                numeric_conflicts[col] += 1

print("\nPRVI conflict types by measurement:")

display(
    pd.DataFrame({
        "Measurement": prvi_measurement_cols,
        "Missingness_Conflicts": [
            missingness_conflicts[col]
            for col in prvi_measurement_cols
        ],
        "Both_Values_Present_Conflicts": [
            numeric_conflicts[col]
            for col in prvi_measurement_cols
        ]
    })
)

### Findings: Duplicate-Record Classification

The duplicate analysis identified 335 duplicate participant-timestamp groups across EI, SII, and PRVI.

| Sensor | Completely identical groups | Validity-flag conflicts | Measurement conflicts |
|---|---:|---:|---:|
| EI | 56 | 14 | 0 |
| SII | 58 | 12 | 0 |
| PRVI | 26 | 0 | 169 |

**EI and SII**

- EI contained 56 completely identical groups and 14 groups with conflicting validity flags.
- SII contained 58 completely identical groups and 12 groups with conflicting validity flags.
- Neither sensor had measurement conflicts.
- In the flag-conflict groups, the measurement values were identical, but `data_validity` differed.

**PRVI**

- PRVI contained 26 completely identical groups and 169 groups with measurement conflicts.
- `mean_HR`, `std_RR`, and `RMSSD` differed numerically in all 169 conflicting groups.
- Several range measurements had a value in one record and a missing value in the other.
- `nRMSSD` had numerical differences in 164 groups and missingness conflicts in five groups.
- The two `range_nRMSSD` fields had missingness conflicts in 164 groups.

**Interpretation and decision**

EI and SII contain exact duplicates as well as records with conflicting validity metadata. PRVI contains a more complex pattern of duplicate records with numerical and missingness differences.

The cause of the PRVI differences is not established. No records were deleted, averaged, imputed, or modified during this investigation. Further analysis is required before selecting a deduplication policy.

## 9. Quantifying PRVI Duplicate Differences

### Objective

Measure the numerical disagreement between duplicate PRVI records and examine their relative completeness.

### Analysis

1. Calculate absolute differences for measurements where both duplicate records have non-missing values.
2. Summarize the mean, median, maximum, and distribution of these differences.
3. Compare the number of available measurement values in each duplicate record.
4. Determine whether one record is consistently more complete than the other.

This analysis will help establish whether the duplicate records are nearly identical, substantially different, or systematically complementary.

In [ ]:
# Step 1: Select PRVI duplicate records

df = sensor_working["PRVI"]

key_cols = ["Participant_ID", "Timestamp"]
flag_cols = ["index_validity", "data_validity"]

measurement_cols = [
    col for col in df.select_dtypes(include="number").columns
    if col not in flag_cols
]

duplicate_rows = df[
    df.duplicated(subset=key_cols, keep=False)
].copy()

prvi_groups = duplicate_rows.groupby(key_cols, sort=False, dropna=False)


# Step 2: Calculate absolute numerical differences

numeric_differences = []
completeness_results = []

for (participant, timestamp), group in prvi_groups:

    group = group.reset_index(drop=True)

    if len(group) != 2:
        continue

    first = group.iloc[0]
    second = group.iloc[1]

    # Compare fields only when both records have values
    for col in measurement_cols:

        if pd.notna(first[col]) and pd.notna(second[col]):

            numeric_differences.append({
                "Measurement": col,
                "Absolute_Difference": abs(first[col] - second[col])
            })

    # Compare the number of available measurement values
    first_count = first[measurement_cols].notna().sum()
    second_count = second[measurement_cols].notna().sum()

    completeness_results.append({
        "Participant_ID": participant,
        "Timestamp": timestamp,
        "First_Record_Values": first_count,
        "Second_Record_Values": second_count,
        "Difference": first_count - second_count
    })


# Step 3: Summarize numerical differences

differences_df = pd.DataFrame(numeric_differences)

print("Absolute numerical differences in PRVI duplicate records:")

display(
    differences_df.groupby("Measurement")["Absolute_Difference"]
    .agg(["count", "mean", "median", "max"])
    .round(8)
)


# Step 4: Compare record completeness

completeness_df = pd.DataFrame(completeness_results)

first_more_complete = (
    completeness_df["Difference"] > 0
).sum()

second_more_complete = (
    completeness_df["Difference"] < 0
).sum()

equally_complete = (
    completeness_df["Difference"] == 0
).sum()

print("\nDuplicate record completeness comparison:")
print("First record more complete:", first_more_complete)
print("Second record more complete:", second_more_complete)
print("Both equally complete:", equally_complete)

print("\nFirst few completeness comparisons:")
display(completeness_df.head(10))


# Step 5: Inspect the distribution of completeness differences

print("\nDistribution of completeness differences:")

display(
    completeness_df["Difference"].value_counts().sort_index()
)

## 10. Tracing PRVI Duplicates to Original Source Files

### Objective

Identify the original PRVI CSV files available in the raw dataset directory.

The purpose is to trace duplicate participant-timestamp records back to their original source files and determine whether the duplicates were already present before standardization.

No source files or standardized records will be modified during this investigation.

In [ ]:
# Locate original PRVI CSV files

raw_path = project_path / "data" / "raw"

print("Raw data directory:", raw_path)
print("Directory exists:", raw_path.exists())

if raw_path.exists():

    prvi_files = list(raw_path.rglob("*PRVI*.csv"))

    print("\nNumber of PRVI CSV files found:", len(prvi_files))

    print("\nFirst 20 matching files:")

    for file in prvi_files[:20]:
        print(file.relative_to(project_path))

else:
    print("Raw data directory was not found at the expected location.")

### Findings: Original PRVI File Availability

- The raw data directory exists at `data/raw`.
- A total of 65 original PRVI CSV files were found.
- The files are stored under the participant-specific directories in `TOLIFE_COPD_Dataset/index`.
- This matches the 65 participants represented in the standardized PRVI dataset.

**Next investigation:** Compare duplicate PRVI records in the standardized dataset with their corresponding original participant CSV records to determine whether the duplicates existed before standardization.

## 11. Comparing Standardized PRVI Duplicates with Raw Source Data

### Objective

Inspect the original PRVI CSV file for participant GER-003 and compare it with a known duplicate timestamp in the standardized dataset.

The comparison will establish whether the duplicate timestamp and its differing measurement values are already present in the raw source file.

This is an initial case study; the result will not automatically be generalized to all PRVI duplicate groups.

In [ ]:
# Step 1: Locate GER-003's original PRVI file

raw_prvi_file = (
    raw_path
    / "TOLIFE_COPD_Dataset"
    / "index"
    / "GERMANY"
    / "GER-003"
    / "PRVI.csv"
)

raw_prvi = pd.read_csv(raw_prvi_file)

print("Original GER-003 PRVI file")
print("Shape:", raw_prvi.shape)
print("\nColumns:")
print(raw_prvi.columns.tolist())

print("\nFirst five rows:")
display(raw_prvi.head())


# Step 2: Parse timestamps as UTC

raw_prvi["Timestamp"] = pd.to_datetime(
    raw_prvi["Timestamp"],
    utc=True,
    errors="coerce"
)

# Step 3: Find the timestamp we previously identified

target_timestamp = pd.Timestamp(
    "2024-10-01 04:42:35.095",
    tz="UTC"
)

raw_matching = raw_prvi[
    raw_prvi["Timestamp"] == target_timestamp
]

print("\nMatching rows in original raw file:", len(raw_matching))
display(raw_matching)


# Step 4: Compare with standardized records

standardized_matching = sensor_working["PRVI"][
    (sensor_working["PRVI"]["Participant_ID"] == "GER-003") &
    (sensor_working["PRVI"]["Timestamp"] == target_timestamp)
]

print("\nMatching rows in standardized dataset:", len(standardized_matching))
display(standardized_matching)

### Findings: GER-003 Raw-to-Standardized Comparison

**Observed results:**

- GER-003's original PRVI CSV contains two rows at `2024-10-01 04:42:35.095 UTC`.
- The standardized PRVI dataset also contains two rows at the same timestamp.
- The measurement values, missing range values, and validity flags match between the raw and standardized records.

**Conclusion:** This duplicate and its differing measurements were already present in the original source file. The standardization process preserved both records rather than creating the duplicate.

**Limitation:** This comparison covers one duplicate group only. The remaining PRVI duplicate groups must be checked before drawing a dataset-wide conclusion.

## 12. Verifying PRVI Duplicates Across All Participants

### Objective

Compare duplicate participant-timestamp keys in every original PRVI CSV file with the standardized PRVI dataset.

For each participant, check:
- Number of duplicate timestamp groups in the raw CSV.
- Number of duplicate timestamp groups in the standardized dataset.
- Whether the duplicate timestamp keys match between the two versions.

This checks whether the duplicate groups were already present in the raw source files. It does not yet decide which duplicate records, if any, should be retained.

In [ ]:
# Build a comparison of duplicate timestamp groups
# between raw PRVI CSV files and standardized PRVI data.

raw_duplicate_summary = {}

for file in raw_path.rglob("PRVI.csv"):

    participant_id = file.parent.name

    raw_df = pd.read_csv(file)

    raw_df["Timestamp"] = pd.to_datetime(
        raw_df["Timestamp"],
        utc=True,
        errors="coerce"
    )

    raw_duplicate_timestamps = set(
        raw_df.loc[
            raw_df.duplicated(subset=["Timestamp"], keep=False),
            "Timestamp"
        ].dropna()
    )

    raw_duplicate_summary[participant_id] = raw_duplicate_timestamps


# Get duplicate timestamp groups from standardized PRVI data

standardized_prvi = sensor_working["PRVI"]

standardized_duplicate_summary = {}

for participant_id, group in standardized_prvi.groupby("Participant_ID"):

    duplicate_timestamps = set(
        group.loc[
            group.duplicated(subset=["Timestamp"], keep=False),
            "Timestamp"
        ].dropna()
    )

    standardized_duplicate_summary[participant_id] = duplicate_timestamps


# Compare raw and standardized duplicate timestamp keys

comparison_rows = []

all_participants = sorted(
    set(raw_duplicate_summary) |
    set(standardized_duplicate_summary)
)

for participant_id in all_participants:

    raw_keys = raw_duplicate_summary.get(participant_id, set())
    standardized_keys = standardized_duplicate_summary.get(
        participant_id, set()
    )

    comparison_rows.append({
        "Participant_ID": participant_id,
        "Raw_duplicate_groups": len(raw_keys),
        "Standardized_duplicate_groups": len(standardized_keys),
        "Matching_groups": len(raw_keys & standardized_keys),
        "Raw_only_groups": len(raw_keys - standardized_keys),
        "Standardized_only_groups": len(standardized_keys - raw_keys)
    })


duplicate_comparison = pd.DataFrame(comparison_rows)

display(
    duplicate_comparison[
        duplicate_comparison[
            [
                "Raw_duplicate_groups",
                "Standardized_duplicate_groups"
            ]
        ].sum(axis=1) > 0
    ]
)

print("\nTOTALS")
print(
    duplicate_comparison[
        [
            "Raw_duplicate_groups",
            "Standardized_duplicate_groups",
            "Matching_groups",
            "Raw_only_groups",
            "Standardized_only_groups"
        ]
    ].sum()
)

### Findings: PRVI Duplicate Source Verification

**Observed results:**

A comparison was performed for every participant with an original PRVI CSV file.

| Metric | Result |
|---|---:|
| Raw duplicate timestamp groups | 195 |
| Standardized duplicate timestamp groups | 195 |
| Matching groups | 195 |
| Raw-only groups | 0 |
| Standardized-only groups | 0 |

All 195 duplicate timestamp groups matched between the raw and standardized datasets.

**Conclusion:** The duplicate timestamp groups were present in the original source files. The standardization process neither introduced new duplicate groups nor removed existing ones.

**Limitation:** This comparison verifies duplicate timestamp keys across all participants. A complete row-by-row value comparison across all 195 groups has not yet been performed.

**Preprocessing decision:** Do not modify or delete PRVI duplicates at this stage. A separate, documented resolution policy will be evaluated before creating a cleaned dataset.

## 13. Simulating a PRVI Duplicate-Handling Policy

### Objective

Test a proposed duplicate-handling policy on a separate working copy of the standardized PRVI dataset, without modifying the original standardized data.

### Candidate policy

For each duplicate group identified by `Participant_ID` and `Timestamp`:

1. If one row has more non-missing measurement values, retain the more complete row.
2. If both rows have equal completeness and identical measurement values, retain one copy.
3. If both rows have equal completeness but different measurement values, retain both and mark the group unresolved.

Validity flags and participant metadata are not counted as measurement values when assessing completeness.

An audit table will record the decision for every row in every duplicate group. The candidate dataset will retain all non-duplicate records and apply the policy only to duplicate groups.

**Important:** This is a simulation for evaluation. The standardized source data will not be modified, and no final cleaning decision is being applied to the source files.

In [ ]:
# Step 1: Create a separate copy for testing

prvi_candidate_source = sensor_working["PRVI"].copy(deep=True)

# Identify measurement columns.
# Exclude validity flags and participant/sensor metadata.

excluded_cols = [
    "Timestamp",
    "Participant_ID",
    "Country",
    "Sensor",
    "index_validity",
    "data_validity"
]

measurement_cols = [
    col
    for col in prvi_candidate_source.select_dtypes(include="number").columns
    if col not in excluded_cols
]

print("Measurement columns used for completeness:")
print(measurement_cols)


# Step 2: Identify duplicate groups

group_cols = ["Participant_ID", "Timestamp"]

duplicate_mask = prvi_candidate_source.duplicated(
    subset=group_cols,
    keep=False
)

duplicate_rows = prvi_candidate_source[duplicate_mask]

print("\nTotal duplicate groups:",
      duplicate_rows.groupby(group_cols).ngroups)

print("Total rows belonging to duplicate groups:",
      len(duplicate_rows))


# Step 3: Simulate the proposed policy

# Begin by retaining every non-duplicate row.
candidate_indices = set(
    prvi_candidate_source.index[~duplicate_mask]
)

audit_records = []
group_decisions = []

for (participant_id, timestamp), group in duplicate_rows.groupby(
    group_cols,
    sort=False
):

    # Count non-missing measurement values in each row.
    completeness = group[measurement_cols].notna().sum(axis=1)

    # Safety check: unexpected group sizes are left unresolved.
    if len(group) != 2:

        decision = "UNRESOLVED_UNEXPECTED_GROUP_SIZE"

        for idx in group.index:
            audit_records.append({
                "Original_Row_Index": idx,
                "Participant_ID": participant_id,
                "Timestamp": timestamp,
                "Measurement_Count": completeness.loc[idx],
                "Decision": decision
            })

        candidate_indices.update(group.index)

    else:

        first_idx = group.index[0]
        second_idx = group.index[1]

        first_count = completeness.loc[first_idx]
        second_count = completeness.loc[second_idx]

        # Case A: first row has more measurements.
        if first_count > second_count:

            candidate_indices.add(first_idx)

            first_decision = "RETAIN_MORE_COMPLETE"
            second_decision = "DISCARD_LESS_COMPLETE"

            decision = "FIRST_ROW_MORE_COMPLETE"

        # Case B: second row has more measurements.
        elif second_count > first_count:

            candidate_indices.add(second_idx)

            first_decision = "DISCARD_LESS_COMPLETE"
            second_decision = "RETAIN_MORE_COMPLETE"

            decision = "SECOND_ROW_MORE_COMPLETE"

        else:

            # Equal completeness: compare all measurement values.
            measurements_identical = (
                group[measurement_cols]
                .iloc[0]
                .reset_index(drop=True)
                .equals(
                    group[measurement_cols]
                    .iloc[1]
                    .reset_index(drop=True)
                )
            )

            if measurements_identical:

                # The two measurement records are identical.
                # Retain one copy.
                candidate_indices.add(first_idx)

                first_decision = "RETAIN_IDENTICAL_COPY"
                second_decision = "DISCARD_IDENTICAL_COPY"

                decision = "EQUALLY_COMPLETE_AND_IDENTICAL"

            else:

                # Equal completeness but different measurements:
                # retain both and flag for manual investigation.
                candidate_indices.update(group.index)

                first_decision = "UNRESOLVED_RETAINED"
                second_decision = "UNRESOLVED_RETAINED"

                decision = "EQUALLY_COMPLETE_BUT_DIFFERENT"

        audit_records.extend([
            {
                "Original_Row_Index": first_idx,
                "Participant_ID": participant_id,
                "Timestamp": timestamp,
                "Measurement_Count": first_count,
                "Decision": first_decision
            },
            {
                "Original_Row_Index": second_idx,
                "Participant_ID": participant_id,
                "Timestamp": timestamp,
                "Measurement_Count": second_count,
                "Decision": second_decision
            }
        ])

    group_decisions.append({
        "Participant_ID": participant_id,
        "Timestamp": timestamp,
        "Group_Decision": decision
    })


# Step 4: Build the audit table

prvi_duplicate_audit = pd.DataFrame(audit_records)

prvi_group_audit = pd.DataFrame(group_decisions)


# Step 5: Build the candidate dataset

prvi_candidate = (
    prvi_candidate_source
    .loc[sorted(candidate_indices)]
    .copy(deep=True)
)


# Step 6: Summarize the decisions

print("\nDUPLICATE GROUP DECISIONS")
display(
    prvi_group_audit["Group_Decision"]
    .value_counts()
    .rename_axis("Decision")
    .reset_index(name="Number_of_Groups")
)

print("\nROW-LEVEL DECISIONS")
display(
    prvi_duplicate_audit["Decision"]
    .value_counts()
    .rename_axis("Decision")
    .reset_index(name="Number_of_Rows")
)


# Step 7: Inspect the complete row-level audit

print("\nFULL DUPLICATE AUDIT")
display(
    prvi_duplicate_audit.sort_values(
        ["Participant_ID", "Timestamp", "Original_Row_Index"]
    ).reset_index(drop=True)
)


# Step 8: Compare original and candidate dataset sizes

print("\nDATASET SIZE COMPARISON")
print("Original PRVI rows:", len(prvi_candidate_source))
print("Candidate PRVI rows:", len(prvi_candidate))
print("Rows removed in simulation:",
      len(prvi_candidate_source) - len(prvi_candidate))

print("\nOriginal duplicate groups:",
      duplicate_rows.groupby(group_cols).ngroups)

print("Duplicate groups remaining in candidate:",
      prvi_candidate.duplicated(
          subset=group_cols,
          keep=False
      ).sum() // 2)

print("\nOriginal standardized dataset remains unchanged:",
      len(sensor_working["PRVI"]) == len(prvi_candidate_source))

## 14. Investigating PRVI Records with No Available Measurements

### Objective

Investigate PRVI records in which all 12 measurement columns are missing.

Specifically:
- Count duplicate groups where both records have no measurements.
- Count duplicate groups where at least one record has measurements.
- Count non-duplicate records with no measurements.
- Inspect their validity flags.

This investigation will help determine whether retaining one copy of a completely empty duplicate record is useful.

No records will be deleted or modified during this step.

In [ ]:
# Step 1: Calculate measurement completeness for all PRVI records

prvi_check = prvi_candidate_source.copy(deep=True)

prvi_check["measurement_count"] = (
    prvi_check[measurement_cols].notna().sum(axis=1)
)

print("Total PRVI records:", len(prvi_check))

print("\nRecords by measurement completeness:")
display(
    prvi_check["measurement_count"]
    .value_counts()
    .sort_index()
    .rename_axis("Available_Measurements")
    .reset_index(name="Number_of_Records")
)


# Step 2: Separate duplicate and non-duplicate records

duplicate_mask = prvi_check.duplicated(
    subset=group_cols,
    keep=False
)

duplicate_records = prvi_check[duplicate_mask]
non_duplicate_records = prvi_check[~duplicate_mask]

print("\nDuplicate records with zero measurements:",
      (duplicate_records["measurement_count"] == 0).sum())

print("Non-duplicate records with zero measurements:",
      (non_duplicate_records["measurement_count"] == 0).sum())


# Step 3: Classify each duplicate group

empty_duplicate_groups = []
nonempty_duplicate_groups = []

for (participant_id, timestamp), group in duplicate_records.groupby(
    group_cols
):

    if (group["measurement_count"] == 0).all():

        empty_duplicate_groups.append({
            "Participant_ID": participant_id,
            "Timestamp": timestamp,
            "Data_Validity": group["data_validity"].tolist(),
            "Index_Validity": group["index_validity"].tolist()
        })

    else:

        nonempty_duplicate_groups.append({
            "Participant_ID": participant_id,
            "Timestamp": timestamp,
            "Measurement_Counts": group["measurement_count"].tolist()
        })


empty_duplicate_groups = pd.DataFrame(empty_duplicate_groups)
nonempty_duplicate_groups = pd.DataFrame(nonempty_duplicate_groups)

print("\nDuplicate groups where BOTH rows have zero measurements:",
      len(empty_duplicate_groups))

print("Duplicate groups where AT LEAST ONE row has measurements:",
      len(nonempty_duplicate_groups))


# Step 4: Inspect completely empty duplicate groups

print("\nCompletely empty duplicate groups:")
display(empty_duplicate_groups)


# Step 5: Inspect non-duplicate records with zero measurements

print("\nNon-duplicate records with zero measurements:")

empty_non_duplicates = non_duplicate_records[
    non_duplicate_records["measurement_count"] == 0
]

display(
    empty_non_duplicates[
        [
            "Timestamp",
            "Participant_ID",
            "data_validity",
            "index_validity",
            "measurement_count"
        ]
    ]
)


# Step 6: Summarize validity flags for all empty records

empty_records = prvi_check[
    prvi_check["measurement_count"] == 0
]

print("\nValidity flag combinations among completely empty records:")

display(
    empty_records.groupby(
        ["data_validity", "index_validity"],
        dropna=False
    ).size().reset_index(name="Number_of_Records")
)

### Findings: PRVI Measurement Completeness and Empty Records

**Observed results:**

The 2,917 standardized PRVI records have the following measurement completeness:

| Available measurement fields | Number of records |
|---:|---:|
| 0 | 505 |
| 4 | 169 |
| 9 | 173 |
| 12 | 2,070 |

All 505 records with zero available measurements have:
- `data_validity = 1`
- `index_validity = NaN`

Among these 505 records:
- 52 rows belong to 26 duplicate timestamp groups, with both records in each group having zero measurements.
- 453 records have no duplicate timestamp.

**Interpretation:**

The `data_validity` flag alone does not establish that usable PRVI measurements are available. Measurement completeness must be assessed independently.

**Proposed handling:**

- Retain one copy from each completely empty duplicate group.
- Retain standalone records with no available measurements.
- Preserve the original validity flags and missing values.
- Do not replace missing measurements with zero.
- Consider adding a separate measurement-availability indicator during feature engineering.

No original or standardized records were modified.

## 15. Final Validation of the PRVI Duplicate-Handling Candidate

### Objective

Validate the simulated PRVI dataset after applying the proposed duplicate-handling policy.

Check the final row count, duplicate timestamps, measurement completeness, and retention of completely empty records.

The candidate remains a separate working copy. The standardized source dataset is not modified.

In [ ]:
# Final validation of the PRVI candidate

print("Original rows:", len(prvi_candidate_source))
print("Candidate rows:", len(prvi_candidate))

# Check duplicate participant-timestamp keys
remaining_duplicates = prvi_candidate.duplicated(
    subset=["Participant_ID", "Timestamp"],
    keep=False
).sum()

print("\nRemaining duplicate rows:", remaining_duplicates)

# Recalculate measurement completeness
candidate_completeness = (
    prvi_candidate[measurement_cols].notna().sum(axis=1)
)

print("\nCandidate measurement completeness:")
display(
    candidate_completeness.value_counts()
    .sort_index()
    .rename_axis("Available_Measurements")
    .reset_index(name="Number_of_Records")
)

# Count completely empty records
empty_candidate = candidate_completeness == 0

print("\nCompletely empty candidate records:",
      empty_candidate.sum())

print("Standalone empty records retained:",
      ((prvi_candidate_source.loc[~duplicate_mask, measurement_cols]
        .isna().all(axis=1)).sum()))

# Check validity flags among empty candidate records
print("\nValidity flags among empty candidate records:")
display(
    prvi_candidate.loc[
        empty_candidate,
        ["data_validity", "index_validity"]
    ].value_counts(dropna=False).reset_index(name="Records")
)

# Confirm original data remains unchanged
print("\nOriginal standardized row count:",
      len(sensor_working["PRVI"]))

print("Original standardized duplicate groups:",
      sensor_working["PRVI"].duplicated(
          subset=["Participant_ID", "Timestamp"],
          keep=False
      ).sum() // 2)

## 16. PRVI Duplicate-Handling Simulation: Final Validation

### Objective

Validate the candidate PRVI dataset produced by applying the proposed duplicate-handling policy.

### Observed results

| Metric | Result |
|---|---:|
| Original records | 2,917 |
| Original duplicate groups | 195 |
| Duplicate rows removed in simulation | 195 |
| Candidate records | 2,722 |
| Remaining duplicate groups | 0 |
| Completely empty candidate records | 479 |

### Duplicate-group decisions

| Decision | Groups |
|---|---:|
| First row more complete | 139 |
| Second row more complete | 30 |
| Equally complete and identical | 26 |
| Equally complete but conflicting | 0 |

### Candidate measurement completeness

| Available measurements | Records |
|---:|---:|
| 0 | 479 |
| 9 | 173 |
| 12 | 2,070 |

### Interpretation

The candidate contains no duplicate participant-timestamp keys.

Of the original 505 completely empty records, 26 redundant copies were removed because they belonged to duplicate groups. All 453 standalone empty records were retained, leaving 479 completely empty records.

All completely empty candidate records have `data_validity = 1` and `index_validity = NaN`. These flags and missing measurement values have been preserved.

### Validation conclusion

All validation checks passed. The candidate dataset is internally consistent with the proposed duplicate-handling policy.

**Important:** This remains a simulated candidate. The original standardized PRVI dataset, containing 2,917 records and 195 duplicate groups, remains unchanged. The candidate has not yet been approved as the final cleaned dataset.

## 16. Saving the PRVI Cleaned Candidate and Audit Reports

### Objective

Save the validated PRVI candidate dataset and its duplicate-handling audit reports in `data/processed/cleaned/`.

### Files to be created

- `PRVI.parquet`: candidate dataset containing 2,722 records.
- `PRVI_duplicate_audit.csv`: row-level record of retention and exclusion decisions.
- `PRVI_duplicate_group_summary.csv`: group-level duplicate decisions.

The standardized PRVI source file will remain unchanged. Existing output files will not be overwritten.

### Validation conditions

Before saving, verify that:
- The candidate contains 2,722 records.
- No duplicate participant-timestamp keys remain.
- The row-level audit contains 390 decisions.
- The group-level audit contains 195 decisions.

This step saves the candidate produced by the tested policy. It does not imply that missing measurements have been imputed or that the remaining empty records have been removed.

In [ ]:
# Step 1: Define output files

cleaned_path = project_path / "data" / "processed" / "cleaned"
cleaned_path.mkdir(parents=True, exist_ok=True)

prvi_output_file = cleaned_path / "PRVI.parquet"
row_audit_file = cleaned_path / "PRVI_duplicate_audit.csv"
group_audit_file = cleaned_path / "PRVI_duplicate_group_summary.csv"

output_files = [
    prvi_output_file,
    row_audit_file,
    group_audit_file
]


# Step 2: Validate the candidate before saving

assert len(prvi_candidate) == 2722

assert not prvi_candidate.duplicated(
    subset=["Participant_ID", "Timestamp"]
).any()

assert len(prvi_duplicate_audit) == 390
assert len(prvi_group_audit) == 195

assert len(sensor_working["PRVI"]) == 2917

print("All pre-save validation checks passed.")


# Step 3: Prevent accidental overwriting

existing_files = [
    file for file in output_files
    if file.exists()
]

if existing_files:

    print("\nSTOP: These output files already exist:")
    for file in existing_files:
        print(file)

    print("\nNo files were saved or overwritten.")

else:

    # Step 4: Save the candidate dataset
    prvi_candidate.to_parquet(
        prvi_output_file,
        index=False
    )

    # Step 5: Save the row-level audit
    prvi_duplicate_audit.to_csv(
        row_audit_file,
        index=False
    )

    # Step 6: Save the group-level audit
    prvi_group_audit.to_csv(
        group_audit_file,
        index=False
    )

    print("\nAll three files saved successfully.")

    for file in output_files:
        print(file)


# Step 7: Read the Parquet file back and verify it

if prvi_output_file.exists():

    saved_prvi = pd.read_parquet(prvi_output_file)

    print("\nSAVED DATASET VERIFICATION")
    print("Saved rows:", len(saved_prvi))

    print(
        "Remaining duplicate groups:",
        saved_prvi.duplicated(
            subset=["Participant_ID", "Timestamp"],
            keep=False
        ).sum() // 2
    )

    print(
        "Saved dataset matches candidate:",
        saved_prvi.equals(
            prvi_candidate.reset_index(drop=True)
        )
    )

## 17. Tracing EI and SII Duplicate Records to Raw Data

### Objective

Determine whether duplicate participant-timestamp records in the standardized EI and SII datasets were already present in the original raw CSV files.

### Investigation

For each sensor:
- Load the original participant-level CSV files.
- Standardize timestamps to UTC for comparison.
- Identify duplicate participant-timestamp groups.
- Compare raw duplicate groups with standardized duplicate groups.
- Check whether duplicate measurements differ and whether validity flags conflict.

### Important distinction

This is a source-tracing and validation step only. No duplicate records will be removed, modified, averaged, or imputed.

In [ ]:
# Step 1: Locate the original raw sensor files

raw_index_path = (
    project_path
    / "data"
    / "raw"
    / "TOLIFE_COPD_Dataset"
    / "index"
)

print("Raw index folder exists:", raw_index_path.exists())


# Step 2: Load and combine the original participant-level files

raw_sensor_data = {}

for sensor in ["EI", "SII"]:

    raw_files = list(raw_index_path.rglob(f"{sensor}.csv"))

    frames = []

    for file in raw_files:

        df = pd.read_csv(file)

        # Participant ID comes from the participant folder name
        df["Participant_ID"] = file.parent.name

        # Convert timestamps to UTC for consistent comparison
        df["Timestamp"] = pd.to_datetime(
            df["Timestamp"],
            utc=True
        )

        frames.append(df)

    if not frames:
        raise FileNotFoundError(
            f"No original {sensor}.csv files found."
        )

    raw_sensor_data[sensor] = pd.concat(
        frames,
        ignore_index=True
    )

    print(
        f"{sensor}: loaded {len(raw_files)} raw files, "
        f"{len(raw_sensor_data[sensor])} rows"
    )


# Step 3: Define a function to summarize duplicate groups

def duplicate_summary(df):

    duplicate_groups = df[
        df.duplicated(
            subset=["Participant_ID", "Timestamp"],
            keep=False
        )
    ]

    grouped = duplicate_groups.groupby(
        ["Participant_ID", "Timestamp"],
        dropna=False
    )

    measurement_cols = [
        col for col in df.columns
        if col not in [
            "Timestamp",
            "Participant_ID",
            "Country",
            "Sensor",
            "index_validity",
            "data_validity"
        ]
    ]

    measurement_conflicts = 0
    flag_conflicts = 0

    for _, group in grouped:

        # Does at least one measurement differ?
        if (
            group[measurement_cols]
            .nunique(dropna=False)
            .gt(1)
            .any()
        ):
            measurement_conflicts += 1

        # Does either validity flag differ?
        if (
            group[["index_validity", "data_validity"]]
            .nunique(dropna=False)
            .gt(1)
            .any()
        ):
            flag_conflicts += 1

    keys = set(
        zip(
            duplicate_groups["Participant_ID"],
            duplicate_groups["Timestamp"]
        )
    )

    return {
        "duplicate_groups": len(grouped),
        "duplicate_extra_rows": len(duplicate_groups) - len(grouped),
        "measurement_conflicts": measurement_conflicts,
        "flag_conflicts": flag_conflicts,
        "duplicate_keys": keys
    }


# Step 4: Compare raw files with standardized datasets

for sensor in ["EI", "SII"]:

    raw_result = duplicate_summary(
        raw_sensor_data[sensor]
    )

    standardized_result = duplicate_summary(
        sensor_working[sensor]
    )

    raw_keys = raw_result["duplicate_keys"]
    standardized_keys = standardized_result["duplicate_keys"]

    print(f"\n{'=' * 55}")
    print(f"{sensor} DUPLICATE SOURCE COMPARISON")
    print(f"{'=' * 55}")

    print("\nRAW DATA")
    print("Duplicate groups:", raw_result["duplicate_groups"])
    print("Extra duplicate rows:", raw_result["duplicate_extra_rows"])
    print("Measurement-conflict groups:", raw_result["measurement_conflicts"])
    print("Flag-conflict groups:", raw_result["flag_conflicts"])

    print("\nSTANDARDIZED DATA")
    print("Duplicate groups:", standardized_result["duplicate_groups"])
    print("Extra duplicate rows:", standardized_result["duplicate_extra_rows"])
    print("Measurement-conflict groups:", standardized_result["measurement_conflicts"])
    print("Flag-conflict groups:", standardized_result["flag_conflicts"])

    print("\nSOURCE MATCHING")
    print("Matching duplicate groups:", len(raw_keys & standardized_keys))
    print("Raw-only duplicate groups:", len(raw_keys - standardized_keys))
    print("Standardized-only duplicate groups:", len(standardized_keys - raw_keys))

    print(
        "All standardized duplicate groups existed in raw data:",
        raw_keys == standardized_keys
    )

## 18. Investigating EI and SII Validity-Flag Conflicts

### Objective

Inspect duplicate EI and SII records whose validity flags disagree, and locate the dataset documentation describing the meaning of `index_validity` and `data_validity`.

### Investigation

- Identify each conflicting participant-timestamp group.
- Display the validity flags for every record in each group.
- Count conflicts separately for `index_validity` and `data_validity`.
- Search local dataset README files for definitions of the validity flags.

### Why this matters

The measurement values in these duplicate groups are identical, but validity flags may carry quality information. We should understand what the flags represent before deciding whether to retain one row, combine flag information, or preserve a conflict indicator.

No records are modified in this step.

In [ ]:
# Step 1: Inspect duplicate groups with conflicting validity flags

for sensor in ["EI", "SII"]:

    df = raw_sensor_data[sensor]

    duplicate_rows = df[
        df.duplicated(
            subset=["Participant_ID", "Timestamp"],
            keep=False
        )
    ]

    print(f"\n{'=' * 65}")
    print(f"{sensor}: CONFLICTING DUPLICATE GROUPS")
    print(f"{'=' * 65}")

    conflict_count = 0

    for (participant, timestamp), group in duplicate_rows.groupby(
        ["Participant_ID", "Timestamp"]
    ):

        index_conflict = (
            group["index_validity"].nunique(dropna=False) > 1
        )

        data_conflict = (
            group["data_validity"].nunique(dropna=False) > 1
        )

        if index_conflict or data_conflict:

            conflict_count += 1

            print(f"\nParticipant: {participant}")
            print(f"Timestamp: {timestamp}")
            print("index_validity:", group["index_validity"].tolist())
            print("data_validity:", group["data_validity"].tolist())

            print(
                "Conflict type:",
                "index_validity" if index_conflict else "",
                "data_validity" if data_conflict else ""
            )

    print(f"\nTotal conflicting groups: {conflict_count}")


# Step 2: Count the type of flag conflicts

for sensor in ["EI", "SII"]:

    df = raw_sensor_data[sensor]

    duplicate_rows = df[
        df.duplicated(
            subset=["Participant_ID", "Timestamp"],
            keep=False
        )
    ]

    index_conflicts = 0
    data_conflicts = 0
    both_conflicts = 0

    for _, group in duplicate_rows.groupby(
        ["Participant_ID", "Timestamp"]
    ):

        index_conflict = (
            group["index_validity"].nunique(dropna=False) > 1
        )

        data_conflict = (
            group["data_validity"].nunique(dropna=False) > 1
        )

        index_conflicts += index_conflict
        data_conflicts += data_conflict
        both_conflicts += index_conflict and data_conflict

    print(f"\n{sensor} FLAG CONFLICT SUMMARY")
    print("index_validity conflicts:", index_conflicts)
    print("data_validity conflicts:", data_conflicts)
    print("Both flags conflict:", both_conflicts)


# Step 3: Locate README files in the dataset

readme_files = list(
    (project_path / "data" / "raw").rglob("README*")
)

print("\nREADME FILES FOUND:", len(readme_files))

for file in readme_files:
    print(file)


# Step 4: Search README text for validity-flag definitions

keywords = [
    "index_validity",
    "data_validity",
    "validity",
    "valid",
    "quality"
]

for file in readme_files:

    try:
        content = file.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        lines = content.splitlines()

        matching_lines = [
            i for i, line in enumerate(lines)
            if any(
                keyword.lower() in line.lower()
                for keyword in keywords
            )
        ]

        if matching_lines:

            print(f"\n{'=' * 65}")
            print("README:", file)
            print(f"{'=' * 65}")

            printed_lines = set()

            for i in matching_lines:

                # Show nearby lines for context
                for j in range(
                    max(0, i - 2),
                    min(len(lines), i + 3)
                ):

                    if j not in printed_lines:
                        print(f"{j + 1}: {lines[j]}")
                        printed_lines.add(j)

    except Exception as error:
        print(f"Could not read {file}: {error}")

## 19. Examining Measurements in EI and SII Flag-Conflict Groups

### Objective

Determine whether duplicate groups with conflicting `data_validity` flags contain zero measurements or non-zero measurements, and search the wider project directory for dataset documentation.

### Investigation

- Examine the measurement columns in every conflicting duplicate group.
- Check whether both duplicate rows have identical measurements.
- Count how many conflicting groups contain all-zero measurements.
- Search the project directory for README files, excluding common environment and Git folders.

### Important distinction

The observed flag conflict does not establish which flag value is correct. Even if the measurements are all zero and `index_validity` is zero, we will preserve the original flag information until a handling policy is justified.

No source or standardized records are modified.

In [ ]:
# Step 1: Examine measurements in every conflicting duplicate group

for sensor in ["EI", "SII"]:

    df = raw_sensor_data[sensor]

    measurement_cols = [
        col for col in df.columns
        if col not in [
            "Timestamp",
            "Participant_ID",
            "Country",
            "Sensor",
            "index_validity",
            "data_validity"
        ]
    ]

    duplicate_rows = df[
        df.duplicated(
            subset=["Participant_ID", "Timestamp"],
            keep=False
        )
    ]

    total_conflicts = 0
    all_zero_groups = 0
    nonzero_groups = 0
    identical_measurement_groups = 0

    print(f"\n{'=' * 65}")
    print(f"{sensor}: MEASUREMENTS IN FLAG-CONFLICT GROUPS")
    print(f"{'=' * 65}")

    for (participant, timestamp), group in duplicate_rows.groupby(
        ["Participant_ID", "Timestamp"]
    ):

        if group["data_validity"].nunique(dropna=False) <= 1:
            continue

        total_conflicts += 1

        measurements = group[measurement_cols]

        # Check whether every measurement in both rows is zero
        all_zero = (
            measurements.notna().all(axis=1)
            & measurements.eq(0).all(axis=1)
        ).all()

        # Check whether the measurement columns are identical
        identical = (
            measurements.nunique(dropna=False).le(1).all()
        )

        if all_zero:
            all_zero_groups += 1
        else:
            nonzero_groups += 1

        if identical:
            identical_measurement_groups += 1

        print(f"\nParticipant: {participant}")
        print(f"Timestamp: {timestamp}")
        print("data_validity:", group["data_validity"].tolist())
        print("index_validity:", group["index_validity"].tolist())
        print("All measurements zero:", all_zero)
        print("Measurements identical:", identical)

        print(
            group[measurement_cols].to_string(index=False)
        )

    print(f"\n{sensor} SUMMARY")
    print("Conflicting groups:", total_conflicts)
    print("All-zero measurement groups:", all_zero_groups)
    print("Groups with non-zero or missing measurements:", nonzero_groups)
    print("Groups with identical measurements:", identical_measurement_groups)


# Step 2: Search the wider project for README files

excluded_folders = {
    ".git",
    ".venv",
    "venv",
    "env",
    "node_modules",
    "__pycache__"
}

readme_files_project = []

for file in project_path.rglob("*"):

    if not file.is_file():
        continue

    if any(part in excluded_folders for part in file.parts):
        continue

    if "readme" in file.name.lower():
        readme_files_project.append(file)

print("\n" + "=" * 65)
print("README FILES FOUND IN PROJECT")
print("=" * 65)

if readme_files_project:
    for file in readme_files_project:
        print(file.relative_to(project_path))
else:
    print("No README files found in the project directory.")

## 20. Checking Project Documentation for Validity-Flag Definitions

### Objective

Inspect the project-level README for any explanation of `index_validity`, `data_validity`, or the source dataset's quality-control conventions.

### Method

Search the README for relevant terms and display matching lines with surrounding context.

### Limitation

A project README may describe the project rather than the original dataset. Unless it explicitly documents the source dataset's flag definitions, we will not treat it as authoritative evidence of what those flags mean.

No data is modified in this step.

In [ ]:
# Locate and inspect the project README

project_readme = project_path / "README.md"

if not project_readme.exists():
    print("Project README.md not found.")
else:
    readme_text = project_readme.read_text(
        encoding="utf-8",
        errors="replace"
    )

    readme_lines = readme_text.splitlines()

    keywords = [
        "index_validity",
        "data_validity",
        "validity",
        "sensor",
        "dataset",
        "tolife",
        "quality"
    ]

    matching_line_numbers = [
        i for i, line in enumerate(readme_lines)
        if any(
            keyword.lower() in line.lower()
            for keyword in keywords
        )
    ]

    print("README:", project_readme)
    print("Total lines:", len(readme_lines))

    if matching_line_numbers:
        displayed_lines = set()

        print("\nRELEVANT README CONTENT\n")

        for i in matching_line_numbers:
            for j in range(
                max(0, i - 2),
                min(len(readme_lines), i + 3)
            ):
                if j not in displayed_lines:
                    print(f"{j + 1}: {readme_lines[j]}")
                    displayed_lines.add(j)
    else:
        print("\nNo relevant keywords found.")

    # If the README is short, display it completely
    if len(readme_lines) <= 100:
        print("\n\nCOMPLETE README\n")
        print(readme_text)

## 21. EI and SII Duplicate-Handling Simulation

### Objective

Simulate duplicate removal for EI and SII while preserving conflicting validity information.

### Proposed rules

1. For duplicate groups with identical measurements and identical validity flags, retain one copy.
2. For duplicate groups with identical measurements but conflicting `data_validity` flags, retain one measurement record, set the cleaned `data_validity` to missing, and add `data_validity_conflict = True`.
3. Preserve both original flag values in the audit records.
4. Retain all non-duplicate records without changing their measurements or flags.

### Rationale

The source does not explain why the conflicting flags occur or which value is correct. Setting the cleaned flag to missing avoids choosing an unsupported value, while the conflict indicator and audit report retain evidence of the disagreement.

### Scope

This is a simulation on independent copies. The standardized source datasets remain unchanged, and no cleaned files are saved in this step.

In [ ]:
# Step 1: Create separate working copies

ei_sii_candidates = {}
ei_sii_row_audits = {}
ei_sii_group_audits = {}

for sensor in ["EI", "SII"]:

    df = sensor_working[sensor].copy(deep=True)

    keys = ["Participant_ID", "Timestamp"]

    measurement_cols = [
        col for col in df.columns
        if col not in [
            "Timestamp",
            "Participant_ID",
            "Country",
            "Sensor",
            "index_validity",
            "data_validity"
        ]
    ]

    keep_indices = []
    conflict_keys = []

    row_audit_records = []
    group_audit_records = []

    # Step 2: Examine each duplicate group

    duplicate_rows = df[
        df.duplicated(subset=keys, keep=False)
    ]

    for (participant, timestamp), group in duplicate_rows.groupby(
        keys,
        sort=False
    ):

        group_indices = list(group.index)
        first_index = group_indices[0]

        # Check whether measurements are identical
        measurements_identical = (
            group[measurement_cols]
            .nunique(dropna=False)
            .le(1)
            .all()
        )

        # Check whether validity flags are identical
        flags_identical = (
            group[["index_validity", "data_validity"]]
            .nunique(dropna=False)
            .le(1)
            .all()
        )

        # Check whether the only difference is data_validity
        data_flag_conflict = (
            group["data_validity"].nunique(dropna=False) > 1
        )

        index_flag_conflict = (
            group["index_validity"].nunique(dropna=False) > 1
        )

        assert measurements_identical
        assert not index_flag_conflict

        keep_indices.append(first_index)

        if flags_identical:

            decision = "IDENTICAL_DUPLICATE"
            conflict_keys_for_group = False

        elif data_flag_conflict:

            decision = "DATA_VALIDITY_CONFLICT"
            conflict_keys.append((participant, timestamp))
            conflict_keys_for_group = True

        else:
            raise ValueError(
                f"Unexpected duplicate pattern: {sensor}, "
                f"{participant}, {timestamp}"
            )

        original_flags = group["data_validity"].tolist()

        group_audit_records.append({
            "Participant_ID": participant,
            "Timestamp": timestamp,
            "Number_of_rows": len(group),
            "Measurements_identical": measurements_identical,
            "Index_validity_values": str(
                group["index_validity"].tolist()
            ),
            "Original_data_validity_values": str(original_flags),
            "Data_validity_conflict": conflict_keys_for_group,
            "Decision": decision
        })

        # Record the decision for every original row
        for index, row in group.iterrows():

            if index == first_index:

                row_decision = (
                    "RETAIN_FLAG_SET_MISSING"
                    if conflict_keys_for_group
                    else "RETAIN_IDENTICAL_COPY"
                )

            else:

                row_decision = (
                    "DISCARD_DUPLICATE_KEEP_FLAG_IN_AUDIT"
                    if conflict_keys_for_group
                    else "DISCARD_IDENTICAL_COPY"
                )

            row_audit_records.append({
                "Original_Row_Index": index,
                "Participant_ID": participant,
                "Timestamp": timestamp,
                "Original_data_validity": row["data_validity"],
                "Original_index_validity": row["index_validity"],
                "Decision": row_decision
            })

    # Step 3: Retain one row per timestamp

    candidate = df.drop_duplicates(
        subset=keys,
        keep="first"
    ).copy()

    # Add an explicit indicator for unresolved flag conflicts
    candidate["data_validity_conflict"] = False

    # Step 4: Mark conflicting data_validity values as missing

    for participant, timestamp in conflict_keys:

        mask = (
            (candidate["Participant_ID"] == participant)
            & (candidate["Timestamp"] == timestamp)
        )

        candidate.loc[mask, "data_validity"] = np.nan

        candidate.loc[
            mask,
            "data_validity_conflict"
        ] = True

    # Step 5: Store the simulated results and audit tables

    ei_sii_candidates[sensor] = candidate

    ei_sii_row_audits[sensor] = pd.DataFrame(
        row_audit_records
    )

    ei_sii_group_audits[sensor] = pd.DataFrame(
        group_audit_records
    )


# Step 6: Display simulation summaries

for sensor in ["EI", "SII"]:

    original = sensor_working[sensor]
    candidate = ei_sii_candidates[sensor]

    row_audit = ei_sii_row_audits[sensor]
    group_audit = ei_sii_group_audits[sensor]

    print(f"\n{'=' * 60}")
    print(f"{sensor} SIMULATION SUMMARY")
    print(f"{'=' * 60}")

    print("Original rows:", len(original))
    print("Candidate rows:", len(candidate))
    print("Rows removed:", len(original) - len(candidate))

    print("\nDuplicate group decisions:")
    print(group_audit["Decision"].value_counts())

    print("\nRow-level decisions:")
    print(row_audit["Decision"].value_counts())

    print(
        "\nRemaining duplicate groups:",
        candidate.duplicated(
            subset=["Participant_ID", "Timestamp"],
            keep=False
        ).sum() // 2
    )

    print(
        "Flag-conflict records:",
        candidate["data_validity_conflict"].sum()
    )

    print(
        "Missing data_validity values:",
        candidate["data_validity"].isna().sum()
    )

    print(
        "Original standardized data unchanged:",
        original.equals(sensor_working[sensor])
    )

## 21. EI and SII Duplicate-Handling Simulation

### Objective

Evaluate a duplicate-handling policy for Environmental Index (EI) and Sound Intensity Index (SII) records.

### Dataset README facts

The TOLIFE dataset documentation defines:
- `index_validity`: indicates whether the calculated health index is valid (1) or invalid (0).
- `data_validity`: indicates whether raw sensor data was available (1) or missing (0).

### Observed analysis results

**EI**
- Original records: 19,732
- Duplicate groups: 70
- Identical measurement groups: 70
- Groups with conflicting `data_validity`: 14

**SII**
- Original records: 19,732
- Duplicate groups: 70
- Identical measurement groups: 70
- Groups with conflicting `data_validity`: 12

All 26 conflicting groups contained identical, entirely zero-valued measurements. Both records in each group had `index_validity = 0`, while their `data_validity` values disagreed.

The simulation produced 19,662 candidate records for each sensor, with no remaining duplicate participant-timestamp groups.

### Duplicate-handling policy evaluated

1. For identical duplicate records with matching flags, retain one copy.
2. For duplicate records with conflicting `data_validity`, retain one measurement record and set the consolidated `data_validity` to missing.
3. Preserve the original conflicting flag values and row-level decisions in the audit report.
4. Preserve `index_validity` without modification.

### Interpretation

Setting `data_validity` to missing represents an unresolved disagreement between duplicate records. It does not establish that the original raw sensor data was unavailable.

The simulation passed its reported validation checks, and the original standardized datasets remained unchanged.

### Limitation

The simulation does not establish why the source records contain conflicting flags or which original flag value is correct. The consolidated records must retain an auditable record of this uncertainty.

## 22. Saving Cleaned EI and SII Datasets

The EI and SII duplicate-handling simulations produced candidate datasets and two levels of audit information.

The results are stored in dictionaries:
- `ei_sii_candidates`: cleaned candidate datasets
- `ei_sii_row_audits`: row-level duplicate decisions
- `ei_sii_group_audits`: group-level duplicate decisions

The candidates and audit reports are saved separately. The original standardized datasets remain unchanged.

Each cleaned dataset is expected to contain 19,662 rows, with no remaining duplicate participant-timestamp combinations.

In [ ]:
# Step 1: Connect the simulation results

simulation_objects = {
    "EI": {
        "candidate": ei_sii_candidates["EI"],
        "row_audit": ei_sii_row_audits["EI"],
        "group_audit": ei_sii_group_audits["EI"]
    },
    "SII": {
        "candidate": ei_sii_candidates["SII"],
        "row_audit": ei_sii_row_audits["SII"],
        "group_audit": ei_sii_group_audits["SII"]
    }
}

# Step 2: Define output paths

cleaned_path = project_path / "data" / "processed" / "cleaned"
cleaned_path.mkdir(parents=True, exist_ok=True)

output_files = {}

for sensor in ["EI", "SII"]:
    output_files[sensor] = {
        "parquet": cleaned_path / f"{sensor}.parquet",
        "row_audit": cleaned_path / f"{sensor}_duplicate_audit.csv",
        "group_audit": cleaned_path / f"{sensor}_duplicate_group_summary.csv"
    }

# Step 3: Validate before saving

for sensor, objects in simulation_objects.items():

    candidate = objects["candidate"]

    assert len(candidate) == 19662

    assert not candidate.duplicated(
        subset=["Participant_ID", "Timestamp"]
    ).any()

    assert len(objects["row_audit"]) == 140
    assert len(objects["group_audit"]) == 70

    print(f"{sensor}: pre-save validation passed.")


# Step 4: Prevent accidental overwriting

existing_files = [
    file
    for sensor_files in output_files.values()
    for file in sensor_files.values()
    if file.exists()
]

if existing_files:

    print("\nSTOP: Existing files detected.")
    for file in existing_files:
        print(file)

    print("Nothing was saved.")

else:

    # Step 5: Save datasets and audit reports

    for sensor, objects in simulation_objects.items():

        objects["candidate"].to_parquet(
            output_files[sensor]["parquet"],
            index=False
        )

        objects["row_audit"].to_csv(
            output_files[sensor]["row_audit"],
            index=False
        )

        objects["group_audit"].to_csv(
            output_files[sensor]["group_audit"],
            index=False
        )

        print(f"\n{sensor} files saved.")

        for file in output_files[sensor].values():
            print(file)


# Step 6: Read back and verify

for sensor in ["EI", "SII"]:

    saved_file = output_files[sensor]["parquet"]

    if saved_file.exists():

        saved_df = pd.read_parquet(saved_file)
        candidate = simulation_objects[sensor]["candidate"]

        print(f"\n{'=' * 45}")
        print(f"{sensor} VERIFICATION")
        print(f"{'=' * 45}")

        print("Saved rows:", len(saved_df))

        print(
            "Remaining duplicate rows:",
            saved_df.duplicated(
                subset=["Participant_ID", "Timestamp"],
                keep=False
            ).sum()
        )

        print(
            "Missing data_validity:",
            saved_df["data_validity"].isna().sum()
        )

        print(
            "Matches simulation candidate:",
            saved_df.equals(candidate.reset_index(drop=True))
        )

## 23. Verifying EI and SII Duplicate Audit Reports

### Objective

Verify that the duplicate audit reports preserve the original records and validity-flag disagreements.

### Expected results

- EI: 14 conflicting duplicate groups.
- SII: 12 conflicting duplicate groups.
- Each conflict should retain evidence of both original `data_validity` values (0 and 1).
- The group-level audit should identify the conflicting groups and their handling decisions.

This verification ensures that resolving duplicate timestamps does not destroy the original data-quality evidence.

In [ ]:
for sensor in ["EI", "SII"]:

    row_audit = ei_sii_row_audits[sensor]
    group_audit = ei_sii_group_audits[sensor]

    print("\n" + "=" * 60)
    print(f"{sensor} AUDIT REPORT INSPECTION")
    print("=" * 60)

    print("\nRow audit columns:")
    print(row_audit.columns.tolist())

    print("\nGroup audit columns:")
    print(group_audit.columns.tolist())

    print("\nRow audit sample:")
    display(row_audit.head(5))

    print("\nGroup audit sample:")
    display(group_audit.head(5))

    # Display rows whose decisions mention flag conflicts
    flag_mask = row_audit.astype(str).apply(
        lambda column: column.str.contains(
            "FLAG",
            case=False,
            na=False
        )
    ).any(axis=1)

    print("\nRows mentioning flag conflicts:")
    display(row_audit[flag_mask])

### Results and Interpretation

The EI and SII audit inspections confirmed that both original validity-flag values are preserved for every conflicting duplicate group.

**Observed results:**
- EI: 14 groups with conflicting `data_validity` values.
- SII: 12 groups with conflicting `data_validity` values.
- Every conflicting group has `index_validity = 0` in both original records.
- The row-level audit preserves both original records, their flags, and their handling decisions.
- The group-level audit records the original flag values and identifies the conflict.

**Cleaning decision:**

For conflicting duplicate groups, one measurement record was retained, its consolidated `data_validity` was set to missing (`NaN`), and the original flag values were preserved in the audit reports.

**Interpretation:**

The disagreement is limited to the `data_validity` flag; the measurements in these duplicate pairs are identical. We cannot determine from the available data why the flag values disagree.

The original standardized datasets remain unchanged.

## 24. Missing Measurement Analysis by Participant

### Objective

Investigate missing measurement values in BRI, MI, PEI, SHRI and SQI.

### Method

For each sensor:
- Identify measurement columns containing missing values.
- Calculate the number and percentage of missing values per measurement.
- Count the number of participants affected by missing values.
- Identify participants with missing measurements.

Timestamp, participant metadata and validity flags are excluded from this measurement-specific analysis.

### Why this matters

Missing values may have different causes. They can represent isolated missing observations, participant-specific recording gaps, or structural differences in which measurements were collected.

Understanding these patterns is necessary before selecting any imputation or exclusion strategy.

No values are modified or imputed during this step.

In [ ]:
# Step 1: Select the five sensors

sensors_to_check = ["BRI", "MI", "PEI", "SHRI", "SQI"]

# Columns that are not measurement variables
non_measurement_columns = [
    "Timestamp",
    "Participant_ID",
    "Country",
    "Sensor",
    "index_validity",
    "data_validity"
]

missingness_results = {}
participant_missingness_results = {}

# Step 2: Analyze each sensor

for sensor in sensors_to_check:

    df = sensor_working[sensor].copy()

    measurement_columns = [
        col for col in df.columns
        if col not in non_measurement_columns
    ]

    # Missing values per measurement column
    missing_counts = df[measurement_columns].isna().sum()

    total_rows = len(df)

    column_summary = pd.DataFrame({
        "missing_count": missing_counts,
        "missing_percentage": (
            missing_counts / total_rows * 100
        ).round(2)
    })

    column_summary = column_summary[
        column_summary["missing_count"] > 0
    ].sort_values("missing_count", ascending=False)

    missingness_results[sensor] = column_summary

    # Step 3: Count participants affected per measurement

    participant_rows = []

    for column in measurement_columns:

        missing_by_participant = (
            df.groupby("Participant_ID")[column]
            .apply(lambda values: values.isna().sum())
        )

        affected = missing_by_participant[
            missing_by_participant > 0
        ]

        participant_rows.append({
            "measurement": column,
            "participants_affected": len(affected),
            "total_participants": df["Participant_ID"].nunique(),
            "participants_with_no_missing": (
                df["Participant_ID"].nunique() - len(affected)
            )
        })

    participant_summary = pd.DataFrame(participant_rows)

    participant_missingness_results[sensor] = participant_summary

    # Step 4: Display results

    print("\n" + "=" * 65)
    print(f"{sensor} — MISSING MEASUREMENT SUMMARY")
    print("=" * 65)

    if column_summary.empty:
        print("No missing measurement values found.")
    else:
        display(column_summary)

    print("\nParticipants affected per measurement:")
    display(participant_summary)

## 25. Investigating Missingness Patterns in BRI, PEI and SQI

### Objective

Investigate the specific distribution of missing measurements identified in Step 24.

### Analysis

- PEI: Compare missing-value masks for HR and SpO2, and for mean_FEV1 and mean_PEF.
- SQI: Identify participants with available values for `use_time` and `duration_movements`.
- BRI: Locate the participant and timestamp associated with missing respiratory-rate statistics.

### Why this matters

Matching missing-value patterns may indicate that measurements were unavailable together. SQI columns missing across nearly the entire dataset may reflect differences in participant-level file schemas rather than isolated missing observations.

Identifying the affected BRI observation will help determine whether its missing values are isolated or associated with a particular recording.

No values are modified, deleted or imputed in this step.

In [ ]:
# ============================================
# PART 1: PEI — Compare paired missing values
# ============================================

pei = sensor_working["PEI"].copy()

paired_columns = [
    ("HR", "SpO2"),
    ("mean_FEV1", "mean_PEF")
]

print("=" * 60)
print("PEI: PAIRED MISSINGNESS ANALYSIS")
print("=" * 60)

for col1, col2 in paired_columns:

    mask1 = pei[col1].isna()
    mask2 = pei[col2].isna()

    print(f"\n{col1} vs {col2}")

    print("Both missing:", (mask1 & mask2).sum())
    print("Only first missing:", (mask1 & ~mask2).sum())
    print("Only second missing:", (~mask1 & mask2).sum())
    print("Neither missing:", (~mask1 & ~mask2).sum())

    print(
        "Missingness masks exactly identical:",
        mask1.equals(mask2)
    )

    if not mask1.equals(mask2):

        print("\nRows with mismatched missingness:")
        display(
            pei.loc[
                mask1 != mask2,
                ["Participant_ID", "Timestamp", col1, col2]
            ]
        )


# ============================================
# PART 2: SQI — Identify available participants
# ============================================

sqi = sensor_working["SQI"].copy()

print("\n" + "=" * 60)
print("SQI: PARTICIPANT-LEVEL AVAILABILITY")
print("=" * 60)

sqi_availability = (
    sqi.groupby("Participant_ID")
    .agg(
        use_time_available=("use_time", lambda x: x.notna().sum()),
        duration_movements_available=(
            "duration_movements",
            lambda x: x.notna().sum()
        ),
        total_records=("Timestamp", "count")
    )
    .reset_index()
)

sqi_availability["use_time_present"] = (
    sqi_availability["use_time_available"] > 0
)

sqi_availability["duration_movements_present"] = (
    sqi_availability["duration_movements_available"] > 0
)

display(
    sqi_availability[
        (sqi_availability["use_time_present"]) |
        (sqi_availability["duration_movements_present"])
    ]
)

print(
    "\nParticipants with use_time:",
    sqi_availability["use_time_present"].sum()
)

print(
    "Participants with duration_movements:",
    sqi_availability["duration_movements_present"].sum()
)


# ============================================
# PART 3: BRI — Locate missing observations
# ============================================

bri = sensor_working["BRI"].copy()

print("\n" + "=" * 60)
print("BRI: MISSING OBSERVATION DETAILS")
print("=" * 60)

bri_measurements = ["rf_mean", "rf_std", "rf_max", "rf_min"]

bri_missing_rows = bri[
    bri[bri_measurements].isna().any(axis=1)
]

display(
    bri_missing_rows[
        ["Participant_ID", "Timestamp"] + bri_measurements
    ]
)

## 26. Comparing Missing Measurements with Validity Flags

### Objective

Determine whether the missing measurements identified in BRI, PEI and SQI correspond to invalid or unavailable sensor data.

### Method

Inspect `data_validity` and `index_validity` for:
- The missing BRI observation.
- PEI observations with missing HR and SpO2.
- PEI observations with missing mean_FEV1 and mean_PEF.
- SQI observations where `use_time` and `duration_movements` are available or missing.

### Why this matters

Missing measurement values and invalidity flags are different concepts. Their relationship can help us understand data quality, but one must not automatically be used to replace or infer the other.

No data is modified in this step.

In [ ]:
# ============================================
# PART 1: BRI — Inspect the missing observation
# ============================================

bri = sensor_working["BRI"].copy()

bri_missing = bri[
    bri[["rf_std", "rf_max", "rf_min"]].isna().any(axis=1)
]

print("=" * 60)
print("BRI: MISSING OBSERVATION AND VALIDITY FLAGS")
print("=" * 60)

display(
    bri_missing[
        [
            "Participant_ID",
            "Timestamp",
            "rf_mean",
            "rf_std",
            "rf_max",
            "rf_min",
            "data_validity",
            "index_validity"
        ]
    ]
)


# ============================================
# PART 2: PEI — Inspect paired missingness
# ============================================

pei = sensor_working["PEI"].copy()

pei_pairs = {
    "HR_SpO2": ["HR", "SpO2"],
    "FEV1_PEF": ["mean_FEV1", "mean_PEF"]
}

print("\n" + "=" * 60)
print("PEI: MISSING MEASUREMENTS AND VALIDITY FLAGS")
print("=" * 60)

for pair_name, columns in pei_pairs.items():

    missing_mask = pei[columns].isna().all(axis=1)

    subset = pei.loc[
        missing_mask,
        ["Participant_ID", "Timestamp"] +
        columns +
        ["data_validity", "index_validity"]
    ]

    print(f"\n{pair_name}")

    print("Missing observations:", len(subset))

    print("\nValidity flag combinations:")

    display(
        subset.groupby(
            ["data_validity", "index_validity"],
            dropna=False
        ).size().reset_index(name="record_count")
    )


# ============================================
# PART 3: SQI — Inspect GER-004 records
# ============================================

sqi = sensor_working["SQI"].copy()

ger004 = sqi[sqi["Participant_ID"] == "GER-004"].copy()

print("\n" + "=" * 60)
print("SQI: GER-004 ADDITIONAL MEASUREMENTS")
print("=" * 60)

print("Total GER-004 records:", len(ger004))

print("\nRecords with additional measurements:")
display(
    ger004[
        ger004[
            ["use_time", "duration_movements"]
        ].notna().any(axis=1)
    ][
        [
            "Timestamp",
            "use_time",
            "duration_movements",
            "data_validity",
            "index_validity"
        ]
    ]
)

print("\nValidity combinations across all GER-004 records:")

display(
    ger004.groupby(
        ["data_validity", "index_validity"],
        dropna=False
    ).size().reset_index(name="record_count")
)


## 27. PEI Missingness Distribution by Participant

### Objective

Investigate how missing PEI measurements are distributed across participants and time.

### Method

For the paired measurement groups:
- HR and SpO2
- mean_FEV1 and mean_PEF

Calculate missing observation counts and missing percentages for each participant. Also identify the earliest and latest timestamps with missing observations.

### Why this matters

Participant-level and temporal concentration can reveal whether missingness is isolated, widespread, or concentrated in particular monitoring periods.

This analysis will help determine whether missingness should be handled as an observation-level issue or considered when assessing participant coverage.

No measurements are modified or imputed.

In [ ]:
pei = sensor_working["PEI"].copy()

pei["Date"] = pei["Timestamp"].dt.date

measurement_pairs = {
    "HR_SpO2": ["HR", "SpO2"],
    "FEV1_PEF": ["mean_FEV1", "mean_PEF"]
}

for pair_name, columns in measurement_pairs.items():

    print("\n" + "=" * 60)
    print(f"PEI: {pair_name} MISSINGNESS BY PARTICIPANT")
    print("=" * 60)

    missing_mask = pei[columns].isna().all(axis=1)

    missing_rows = pei[missing_mask].copy()

    # Participant-level summary
    participant_summary = (
        pei.assign(missing=missing_mask)
        .groupby("Participant_ID")
        .agg(
            total_records=("Timestamp", "count"),
            missing_records=("missing", "sum")
        )
        .reset_index()
    )

    participant_summary["missing_percentage"] = (
        participant_summary["missing_records"] /
        participant_summary["total_records"] * 100
    ).round(2)

    participant_summary = participant_summary[
        participant_summary["missing_records"] > 0
    ].sort_values("missing_records", ascending=False)

    print("\nParticipants with missing observations:")
    display(participant_summary)

    print("\nOverall missingness period:")

    if len(missing_rows) > 0:
        print("Earliest:", missing_rows["Timestamp"].min())
        print("Latest:", missing_rows["Timestamp"].max())

    print("\nParticipants affected:", missing_rows["Participant_ID"].nunique())
    print("Total missing observations:", len(missing_rows))
    

In [ ]:
# ============================================
# PEI: CONSECUTIVE MISSINGNESS ANALYSIS
# ============================================

pei = sensor_working["PEI"].copy()

pei["Timestamp"] = pd.to_datetime(
    pei["Timestamp"], utc=True
)

pei = pei.sort_values(
    ["Participant_ID", "Timestamp"]
).reset_index(drop=True)


def find_missing_runs(df, column, pair_name):

    results = []

    for participant, group in df.groupby("Participant_ID"):

        group = group.sort_values("Timestamp").copy()

        # True means the selected measurement is missing
        group["is_missing"] = group[column].isna()

        # Create a new run whenever missing status changes
        group["run_id"] = (
            group["is_missing"] != group["is_missing"].shift()
        ).cumsum()

        missing_groups = group[group["is_missing"]].groupby("run_id")

        for _, run in missing_groups:

            results.append({
                "measurement_pair": pair_name,
                "Participant_ID": participant,
                "missing_records": len(run),
                "first_missing": run["Timestamp"].min(),
                "last_missing": run["Timestamp"].max(),
                "calendar_span_days": (
                    run["Timestamp"].max() -
                    run["Timestamp"].min()
                ).total_seconds() / 86400
            })

    return pd.DataFrame(results)


# Analyze the two measurement pairs

hr_runs = find_missing_runs(
    pei,
    "HR",
    "HR_SpO2"
)

fev1_runs = find_missing_runs(
    pei,
    "mean_FEV1",
    "FEV1_PEF"
)


# ============================================
# DISPLAY RESULTS
# ============================================

for pair_name, runs in [
    ("HR / SpO2", hr_runs),
    ("mean_FEV1 / mean_PEF", fev1_runs)
]:

    print("\n" + "=" * 65)
    print(f"{pair_name}: CONSECUTIVE MISSINGNESS")
    print("=" * 65)

    if runs.empty:
        print("No missing runs found.")
        continue

    # Longest missing run for each participant
    longest_runs = (
        runs.sort_values(
            ["Participant_ID", "missing_records"],
            ascending=[True, False]
        )
        .groupby("Participant_ID")
        .head(1)
        .sort_values(
            "missing_records",
            ascending=False
        )
    )

    display(longest_runs.head(15))

    print("\nOverall run statistics:")

    print("Total missing runs:", len(runs))
    print("Longest run:", runs["missing_records"].max())
    print("Median run length:", runs["missing_records"].median())

    print(
        "Runs containing 2 or more records:",
        (runs["missing_records"] >= 2).sum()
    )

## 28. Missing-Value Handling Policy

### Objective

Define a conservative missing-value handling policy based on the missingness analysis.

### Decisions

**BRI**
- Preserve the three missing respiratory-rate statistics.
- Retain the observation because `rf_mean` is available.

**MI and SHRI**
- No missing measurement values were detected.
- No missing-value treatment is required.

**PEI**
- Preserve missing HR and SpO2 values.
- Preserve missing mean_FEV1 and mean_PEF values.
- Keep paired missingness intact.
- Do not impute measurements based solely on adjacent observations.

**SQI**
- Preserve the original missing values in all columns.
- Retain `use_time` and `duration_movements` as structurally sparse fields.
- Do not use these fields as general model predictors without a separate coverage assessment.

### Data preservation

No missing measurements are replaced with zero, interpolated, forward-filled, backward-filled or statistically imputed.

Validity flags remain unchanged.

### Interpretation

Missing measurements are retained as missing because their underlying values cannot be reliably established from the available dataset. Their distribution and coverage can instead be represented through separate data-quality features during later feature engineering.

In [ ]:
# Step 1: Create cleaned working copies

sensors_to_process = ["BRI", "MI", "PEI", "SHRI", "SQI"]

cleaned_remaining = {}

for sensor in sensors_to_process:
    cleaned_remaining[sensor] = (
        sensor_working[sensor].copy(deep=True)
    )


# Step 2: Compare missing values before and after

for sensor in sensors_to_process:

    original = sensor_working[sensor]
    cleaned = cleaned_remaining[sensor]

    print("\n" + "=" * 55)
    print(f"{sensor}: MISSINGNESS PRESERVATION CHECK")
    print("=" * 55)

    original_missing = original.isna().sum()
    cleaned_missing = cleaned.isna().sum()

    comparison = pd.DataFrame({
        "original_missing": original_missing,
        "cleaned_missing": cleaned_missing,
        "unchanged": original_missing == cleaned_missing
    })

    display(comparison)

    assert original.shape == cleaned.shape
    assert original_missing.equals(cleaned_missing)

    print("Row count preserved:", len(cleaned))
    print("Column count preserved:", len(cleaned.columns))
    print("All missing-value counts preserved: True")


print("\nAll five sensor working copies created successfully.")

## 30. Measurement Range and Outlier Investigation

### Objective

Examine the numerical distributions of measurements across all eight sensor datasets.

### Method

For each numerical measurement, calculate:
- Minimum and maximum values.
- 1st and 99th percentiles.
- Median.
- Number of zero values.
- Number of negative values.
- Number of missing values.

Validity flags, timestamps and participant metadata are excluded from the measurement analysis.

### Why this matters

This analysis helps identify unusual values, potential sensor errors, and measurements that may require further investigation.

An extreme value is not automatically an outlier or an error. In a healthcare dataset, unusual values may represent genuine changes in a patient's condition.

### Important limitation

This is an exploratory statistical screening, not clinical validation. No measurements are removed, clipped, transformed or replaced in this step.

In [ ]:

# Combine all eight cleaned sensor datasets
all_cleaned_sensors = {
    "BRI": cleaned_remaining["BRI"],
    "EI": ei_sii_candidates["EI"],
    "MI": cleaned_remaining["MI"],
    "PEI": cleaned_remaining["PEI"],
    "PRVI": prvi_candidate,
    "SHRI": cleaned_remaining["SHRI"],
    "SII": ei_sii_candidates["SII"],
    "SQI": cleaned_remaining["SQI"]
}

non_measurement_columns = [
    "Timestamp", "Participant_ID", "Country", "Sensor",
    "index_validity", "data_validity"
]

range_results = {}
boolean_columns_found = {}

for sensor, df in all_cleaned_sensors.items():

    measurement_columns = [
        col for col in df.columns
        if col not in non_measurement_columns
        and (
            pd.api.types.is_numeric_dtype(df[col])
            or pd.api.types.is_bool_dtype(df[col])
        )
    ]

    records = []
    boolean_columns = []

    for column in measurement_columns:

        values = df[column]

        # Temporarily convert Boolean values for statistical analysis
        if pd.api.types.is_bool_dtype(values):
            boolean_columns.append(column)
            values = values.astype("float64")

        non_missing = values.dropna()

        if len(non_missing) == 0:
            continue

        records.append({
            "measurement": column,
            "original_dtype": str(df[column].dtype),
            "non_missing_count": len(non_missing),
            "missing_count": values.isna().sum(),
            "minimum": non_missing.min(),
            "p01": non_missing.quantile(0.01),
            "median": non_missing.median(),
            "p99": non_missing.quantile(0.99),
            "maximum": non_missing.max(),
            "zero_count": (non_missing == 0).sum(),
            "negative_count": (non_missing < 0).sum()
        })

    range_results[sensor] = pd.DataFrame(records)
    boolean_columns_found[sensor] = boolean_columns

    print(f"\n{'=' * 25} {sensor} {'=' * 25}")
    print("Boolean columns:", boolean_columns)
    display(range_results[sensor])

print("\nBoolean columns found across sensors:")
for sensor, columns in boolean_columns_found.items():
    if columns:
        print(sensor, ":", columns)


### Step 30A: Handling Boolean Columns in Measurement Range Analysis

During the initial measurement range analysis, a `TypeError` occurred because at least one measurement column was stored using a Boolean data type.

Percentile calculations require numerical values and cannot directly interpolate Boolean values.

To address this, Boolean measurement columns are temporarily converted to numeric values (`True` = 1 and `False` = 0) within the analysis. The original cleaned datasets are not modified.

This adjustment is solely for statistical inspection and does not represent a decision to modify or clean the underlying measurements.

### Step 30B: Excluding Data-Quality Audit Columns

The initial range analysis identified `data_validity_conflict` in the EI and SII datasets.

This Boolean column was introduced during duplicate-resolution auditing. It records whether duplicate rows originally had conflicting `data_validity` flags; it is not a sensor measurement.

Therefore, it must be excluded from measurement statistics, alongside timestamps, participant metadata, sensor labels and validity flags.

No underlying measurements are modified.

In [ ]:

non_measurement_columns = [
    "Timestamp",
    "Participant_ID",
    "Country",
    "Sensor",
    "index_validity",
    "data_validity",
    "data_validity_conflict"
]


### Step 31: Measurement Ranges Among Records with Valid Indices

The overall measurement summaries include many zero-valued observations. Previous analysis showed that a substantial proportion of these observations are associated with `index_validity = 0`.

To better understand the measurement distributions, this step calculates descriptive statistics using only records with `index_validity = 1`.

The original datasets remain unchanged. Records with invalid or missing validity flags are excluded only from this particular analysis.

**Important:** A valid index flag does not independently prove that every measurement is clinically plausible. The results are used for investigation, not automatic outlier removal.

In [ ]:

valid_range_results = {}

for sensor, df in all_cleaned_sensors.items():

    valid_df = df[df["index_validity"] == 1].copy()

    measurement_columns = [
        col for col in valid_df.columns
        if col not in non_measurement_columns
        and pd.api.types.is_numeric_dtype(valid_df[col])
    ]

    records = []

    for column in measurement_columns:

        values = valid_df[column].dropna()

        if len(values) == 0:
            continue

        records.append({
            "measurement": column,
            "valid_record_count": len(values),
            "minimum": values.min(),
            "p01": values.quantile(0.01),
            "median": values.median(),
            "p99": values.quantile(0.99),
            "maximum": values.max(),
            "zero_count": (values == 0).sum(),
            "negative_count": (values < 0).sum()
        })

    valid_range_results[sensor] = pd.DataFrame(records)

    print(f"\n{'=' * 25} {sensor} — VALID INDEX RECORDS {'=' * 25}")
    print(f"Total records with index_validity = 1: {len(valid_df)}")
    display(valid_range_results[sensor])


### Step 32: Investigating Suspicious Measurement Values

The valid-index measurement summaries identified several observations that require further investigation.

This step extracts the individual records associated with selected extreme values, including their participant IDs, timestamps and validity flags.

The investigation focuses on negative sleep efficiency values, unusually high sound-intensity variability, unusually low PRVI measurements, zero-valued environmental measurements and extreme PEI measurements.

The purpose is to determine whether these observations are isolated, repeated or associated with other data-quality indicators.

No records are removed, corrected or imputed during this investigation.

In [ ]:

# Helper function to display selected records with relevant metadata

def show_records(df, columns, sort_column=None, ascending=True, n=10):
    available_columns = [
        col for col in columns
        if col in df.columns
    ]

    result = df.copy()

    if sort_column is not None:
        result = result.sort_values(
            sort_column,
            ascending=ascending,
            na_position="last"
        )

    display(result[available_columns].head(n))


metadata_columns = [
    "Participant_ID",
    "Timestamp",
    "index_validity",
    "data_validity"
]

# 1. Negative SQI sleep efficiency
print("\n1. SQI — ALL NEGATIVE SE VALUES")

sqi = all_cleaned_sensors["SQI"]

negative_se = sqi[
    (sqi["index_validity"] == 1) &
    (sqi["SE"] < 0)
].copy()

print("Negative SE records:", len(negative_se))

show_records(
    negative_se,
    metadata_columns + [
        "SE", "sleep_duration", "WASO", "num_movements"
    ],
    sort_column="SE",
    ascending=True,
    n=30
)


# 2. SII — Highest sound intensity variability
print("\n2. SII — HIGHEST Std_SI VALUES")

sii = all_cleaned_sensors["SII"]

sii_valid = sii[sii["index_validity"] == 1].copy()

show_records(
    sii_valid,
    metadata_columns + [
        "Mean_SI", "Std_SI", "Max_SI", "Min_SI"
    ],
    sort_column="Std_SI",
    ascending=False,
    n=10
)


# 3. PRVI — Lowest std_RR values
print("\n3. PRVI — LOWEST std_RR VALUES")

prvi = all_cleaned_sensors["PRVI"]

prvi_valid = prvi[prvi["index_validity"] == 1].copy()

show_records(
    prvi_valid,
    metadata_columns + [
        "mean_HR", "std_RR", "RMSSD", "nRMSSD"
    ],
    sort_column="std_RR",
    ascending=True,
    n=10
)


# 4. EI — Valid-index records with zero environmental measurements
print("\n4. EI — ZERO-VALUED ENVIRONMENTAL MEASUREMENTS")

ei = all_cleaned_sensors["EI"]

ei_measurements = [
    "Mean_Temp", "Max_Temp", "Min_Temp",
    "Mean_Hum", "Max_Hum", "Min_Hum",
    "Mean_AQI", "Max_AQI", "Min_AQI",
    "Mean_Lux", "Max_Lux", "Min_Lux"
]

ei_valid = ei[ei["index_validity"] == 1].copy()

zero_ei = ei_valid[
    (ei_valid[ei_measurements] == 0).any(axis=1)
].copy()

print("Records with at least one zero measurement:", len(zero_ei))

show_records(
    zero_ei,
    metadata_columns + ei_measurements,
    n=30
)


# 5. PEI — Lowest SpO2 values
print("\n5. PEI — LOWEST SpO2 VALUES")

pei = all_cleaned_sensors["PEI"]

pei_valid = pei[pei["index_validity"] == 1].copy()

show_records(
    pei_valid,
    metadata_columns + [
        "SpO2", "HR", "mean_FEV1", "mean_PEF"
    ],
    sort_column="SpO2",
    ascending=True,
    n=10
)


# 6. PEI — Highest heart-rate values
print("\n6. PEI — HIGHEST HR VALUES")

show_records(
    pei_valid,
    metadata_columns + [
        "HR", "SpO2", "mean_FEV1", "mean_PEF"
    ],
    sort_column="HR",
    ascending=False,
    n=10
)


### Step 33: Participant-Level Temporal Investigation of Suspicious Measurements

The previous investigation identified unusual measurements in SQI, SII, PRVI, PEI and EI.

This step examines selected participants over time to determine whether the observations are isolated or recurring.

The analysis focuses on:
- Repeated negative sleep-efficiency values for SPA-019.
- Unusually high sound-intensity measurements for SPA-013.
- Low PRVI standard-deviation-of-RR observations for GER-018 and GER-011.
- SpO2 and heart-rate patterns for SPA-010.
- The individual EI measurement columns responsible for zero-valued observations.

This is an exploratory data-quality investigation. No measurements are removed or modified.

In [ ]:

# 1. SPA-019 — SQI history
print("\n1. SPA-019 — SQI HISTORY")

sqi = all_cleaned_sensors["SQI"]

spa019_sqi = sqi[
    sqi["Participant_ID"] == "SPA-019"
].sort_values("Timestamp")

display(
    spa019_sqi[
        [
            "Timestamp", "index_validity", "data_validity",
            "sleep_duration", "WASO", "num_movements", "SE"
        ]
    ]
)

print("Negative SE count:", (spa019_sqi["SE"] < 0).sum())


# 2. SPA-013 — SII history around October 2024
print("\n2. SPA-013 — SII HISTORY")

sii = all_cleaned_sensors["SII"]

spa013_sii = sii[
    (sii["Participant_ID"] == "SPA-013") &
    (sii["Timestamp"] >= "2024-10-15") &
    (sii["Timestamp"] <= "2024-10-25")
].sort_values("Timestamp")

display(
    spa013_sii[
        [
            "Timestamp", "index_validity", "data_validity",
            "Mean_SI", "Std_SI", "Max_SI", "Min_SI"
        ]
    ]
)


# 3. GER-018 and GER-011 — PRVI history
print("\n3. GER-018 AND GER-011 — PRVI HISTORY")

prvi = all_cleaned_sensors["PRVI"]

for participant in ["GER-018", "GER-011"]:

    print(f"\nParticipant: {participant}")

    participant_data = prvi[
        prvi["Participant_ID"] == participant
    ].sort_values("Timestamp")

    display(
        participant_data[
            [
                "Timestamp", "index_validity", "data_validity",
                "mean_HR", "std_RR", "RMSSD", "nRMSSD"
            ]
        ]
    )


# 4. SPA-010 — PEI history
print("\n4. SPA-010 — PEI HISTORY")

pei = all_cleaned_sensors["PEI"]

spa010_pei = pei[
    pei["Participant_ID"] == "SPA-010"
].sort_values("Timestamp")

display(
    spa010_pei[
        [
            "Timestamp", "index_validity", "data_validity",
            "SpO2", "HR", "mean_FEV1", "mean_PEF"
        ]
    ]
)


# 5. EI — Zero counts by individual measurement column
print("\n5. EI — ZERO COUNTS BY MEASUREMENT")

ei = all_cleaned_sensors["EI"]

ei_measurements = [
    col for col in ei.columns
    if col not in non_measurement_columns
    and pd.api.types.is_numeric_dtype(ei[col])
]

ei_valid = ei[ei["index_validity"] == 1]

ei_zero_summary = pd.DataFrame({
    "measurement": ei_measurements,
    "zero_count": [
        (ei_valid[col] == 0).sum()
        for col in ei_measurements
    ],
    "valid_record_count": [
        ei_valid[col].notna().sum()
        for col in ei_measurements
    ]
})

ei_zero_summary["zero_percentage"] = (
    ei_zero_summary["zero_count"] /
    ei_zero_summary["valid_record_count"] * 100
)

display(ei_zero_summary)


### Step 33: Findings from Participant-Level Temporal Investigation

#### 1. SQI — SPA-019
- The participant has 9 negative sleep-efficiency (`SE`) observations.
- Negative values recur across multiple dates rather than appearing as a single isolated observation.
- These observations have previously been found with both validity flags set to 1.
- Since sleep efficiency is expressed as a percentage, negative values require further investigation.

#### 2. SII — SPA-013
- Two unusually large sound-intensity observations occur on 19 and 20 October 2024.
- The corresponding `Mean_SI` values are 24,825.80 and 23,221.38.
- Several surrounding records have zero measurements and `index_validity = 0`.
- Measurements return to approximately 1,940 on 21 October.
- This is a localized discontinuity that warrants further investigation.

#### 3. PRVI — GER-018 and GER-011
- GER-018 has several relatively low `std_RR` observations during September and October 2024.
- GER-011 has an isolated minimum `std_RR` observation of 60 on 16 August 2024.
- The surrounding observations are substantially higher.
- These observations are retained because unusual physiological measurements cannot be classified as errors using statistical magnitude alone.

#### 4. PEI — SPA-010
- The participant's history contains many records with zero measurements and invalid index flags.
- Some records have valid measurements, including low SpO2 observations.
- Invalid zero-valued observations must be distinguished from valid measurements before calculating trends or patient baselines.

#### 5. EI — Zero-valued measurements
- Zero values are particularly frequent in `Min_Lux`, `Mean_Lux` and `Max_Lux`.
- Zero light measurements may occur under dark environmental conditions.
- Other zero-valued measurements, particularly environmental minima, require interpretation using their definitions and surrounding observations.

#### Overall interpretation
The investigation identified recurring and localized measurement patterns that cannot be resolved using validity flags or statistical thresholds alone.

No measurements were deleted, imputed, or modified. The identified observations will be retained for further quality assessment and may later be represented through data-quality indicators.

### Step 34: Focused Inspection of Anomalous Observations

This step examines the exact dates and surrounding observations associated with the anomalies identified in Step 33.

The investigation focuses on:
- All negative sleep-efficiency observations for SPA-019.
- The two unusual SII records for SPA-013 and their surrounding measurements.
- The isolated low PRVI observation for GER-011 and its neighboring records.
- Valid PEI observations for SPA-010, with particular attention to SpO2.
- EI zero-valued minimum AQI and light measurements.

The objective is to establish whether the observations are isolated, repeated, or associated with abrupt changes in adjacent records.

No modifications are made to the datasets.

In [ ]:

# 1. SPA-019 — exact negative SE records
print("\n1. SPA-019 — NEGATIVE SE RECORDS")

sqi = all_cleaned_sensors["SQI"]

spa019_negative = sqi[
    (sqi["Participant_ID"] == "SPA-019") &
    (sqi["SE"] < 0)
].sort_values("Timestamp")

display(
    spa019_negative[
        [
            "Timestamp", "index_validity", "data_validity",
            "sleep_duration", "WASO", "num_movements", "SE"
        ]
    ]
)


# 2. SPA-013 — SII anomaly and nearby records
print("\n2. SPA-013 — SII ANOMALY WINDOW")

sii = all_cleaned_sensors["SII"]

spa013_window = sii[
    (sii["Participant_ID"] == "SPA-013") &
    (sii["Timestamp"] >= "2024-10-18") &
    (sii["Timestamp"] <= "2024-10-21 23:59:59")
].sort_values("Timestamp")

display(
    spa013_window[
        [
            "Timestamp", "index_validity", "data_validity",
            "Mean_SI", "Std_SI", "Max_SI", "Min_SI"
        ]
    ]
)


# 3. GER-011 — PRVI records around the minimum std_RR
print("\n3. GER-011 — PRVI AROUND 16 AUGUST 2024")

prvi = all_cleaned_sensors["PRVI"]

ger011_window = prvi[
    (prvi["Participant_ID"] == "GER-011") &
    (prvi["Timestamp"] >= "2024-08-10") &
    (prvi["Timestamp"] <= "2024-08-23")
].sort_values("Timestamp")

display(
    ger011_window[
        [
            "Timestamp", "index_validity", "data_validity",
            "mean_HR", "std_RR", "RMSSD", "nRMSSD"
        ]
    ]
)


# 4. SPA-010 — only valid PEI observations
print("\n4. SPA-010 — VALID PEI OBSERVATIONS")

pei = all_cleaned_sensors["PEI"]

spa010_valid = pei[
    (pei["Participant_ID"] == "SPA-010") &
    (pei["index_validity"] == 1)
].sort_values("Timestamp")

print("Total valid-index records:", len(spa010_valid))

display(
    spa010_valid[
        [
            "Timestamp", "data_validity",
            "SpO2", "HR", "mean_FEV1", "mean_PEF"
        ]
    ]
)


# 5. EI — zero-valued minimum AQI and light
print("\n5. EI — ZERO MIN_AQI RECORDS")

ei = all_cleaned_sensors["EI"]

ei_valid = ei[ei["index_validity"] == 1].copy()

zero_min_aqi = ei_valid[ei_valid["Min_AQI"] == 0]

print("Valid-index records with Min_AQI = 0:", len(zero_min_aqi))

display(
    zero_min_aqi[
        [
            "Participant_ID", "Timestamp",
            "data_validity", "Mean_AQI", "Max_AQI", "Min_AQI"
        ]
    ].head(20)
)

print("\nEI — ZERO MIN_LUX RECORDS")

zero_min_lux = ei_valid[ei_valid["Min_Lux"] == 0]

print("Valid-index records with Min_Lux = 0:", len(zero_min_lux))

display(
    zero_min_lux[
        [
            "Participant_ID", "Timestamp",
            "data_validity", "Mean_Lux", "Max_Lux", "Min_Lux"
        ]
    ].head(20)
)


### Step 35: Quantifying Remaining Suspicious Measurements

This step quantifies two observations from the previous investigation:

1. SPA-010's valid PEI measurements, particularly SpO2, to understand their distribution across all available valid-index records.
2. EI records where all four AQI measurements are zero, to distinguish these from records where only the minimum AQI is zero.

The analysis uses existing validity flags and does not apply new clinical thresholds to classify patients.

No measurements are modified or removed.

In [ ]:

# 1. SPA-010 — PEI statistical summary
print("\n1. SPA-010 — PEI SUMMARY")

spa010_summary = spa010_valid[
    ["SpO2", "HR", "mean_FEV1", "mean_PEF"]
].describe().T

spa010_summary["missing_count"] = (
    spa010_valid[
        ["SpO2", "HR", "mean_FEV1", "mean_PEF"]
    ].isna().sum()
)

display(spa010_summary)

print("\nSpO2 distribution:")
print("Below 80:", (spa010_valid["SpO2"] < 80).sum())
print("80 to below 85:", (
    (spa010_valid["SpO2"] >= 80) &
    (spa010_valid["SpO2"] < 85)
).sum())
print("85 or above:", (spa010_valid["SpO2"] >= 85).sum())

print("\nValid-index records with data_validity = 0:")
print((spa010_valid["data_validity"] == 0).sum())


# 2. EI — records where all four AQI measurements are zero
print("\n2. EI — ALL-ZERO AQI RECORDS")

aqi_columns = [
    "Mean_AQI", "Std_AQI", "Max_AQI", "Min_AQI"
]

all_zero_aqi = ei_valid[
    (ei_valid[aqi_columns] == 0).all(axis=1)
]

print("All-zero AQI records:", len(all_zero_aqi))
print(
    "Participants affected:",
    all_zero_aqi["Participant_ID"].nunique()
)

display(
    all_zero_aqi[
        [
            "Participant_ID", "Timestamp",
            "index_validity", "data_validity"
        ] + aqi_columns
    ].sort_values(["Participant_ID", "Timestamp"])
)


# 3. EI — compare all-zero AQI against zero Min_AQI only
print("\n3. EI — AQI ZERO PATTERN COUNTS")

aqi_zero_patterns = pd.Series({
    "All four AQI measurements zero": len(all_zero_aqi),

    "Only Min_AQI zero, other AQI measurements not all zero": len(
        ei_valid[
            (ei_valid["Min_AQI"] == 0) &
            ~((ei_valid[aqi_columns] == 0).all(axis=1))
        ]
    ),

    "No AQI measurement equal to zero": len(
        ei_valid[
            ~(ei_valid[aqi_columns] == 0).any(axis=1)
        ]
    )
})

display(aqi_zero_patterns.to_frame("record_count"))


### Step 36: Creating a Data-Quality Review Audit

A separate data-quality audit is created to document observations that require further review without modifying the original measurements.

The audit includes:
- Negative sleep-efficiency observations in SPA-019.
- All-zero environmental index observations in EI.
- Unusually large sound-intensity observations in SPA-013.

Each audit entry records the participant, timestamp, sensor, affected measurement, observed value, and reason for review.

These flags indicate data-quality concerns, not confirmed measurement errors. Original measurements and validity flags remain unchanged.

Other unusual observations, such as SPA-010's low SpO2 readings and GER-011's isolated low `std_RR`, are retained without automatically classifying them as invalid.

In [ ]:

# Create a separate audit table for suspicious observations

quality_review_records = []


# 1. SPA-019 — negative sleep efficiency
for _, row in spa019_negative.iterrows():

    quality_review_records.append({
        "Participant_ID": row["Participant_ID"],
        "Timestamp": row["Timestamp"],
        "Sensor": "SQI",
        "Measurement": "SE",
        "Observed_Value": row["SE"],
        "Reason": "Negative sleep efficiency percentage",
        "Review_Status": "Requires investigation"
    })


# 2. EI — all four AQI measurements are zero
for _, row in all_zero_aqi.iterrows():

    quality_review_records.append({
        "Participant_ID": row["Participant_ID"],
        "Timestamp": row["Timestamp"],
        "Sensor": "EI",
        "Measurement": "All AQI measurements",
        "Observed_Value": 0,
        "Reason": "All AQI measurements are zero despite valid flags",
        "Review_Status": "Requires investigation"
    })


# 3. SPA-013 — unusually high SII measurements
sii_anomalies = sii[
    (sii["Participant_ID"] == "SPA-013") &
    (sii["Timestamp"] >= "2024-10-19") &
    (sii["Timestamp"] <= "2024-10-20 23:59:59") &
    (sii["Mean_SI"] > 10000) &
    (sii["index_validity"] == 1)
]

for _, row in sii_anomalies.iterrows():

    quality_review_records.append({
        "Participant_ID": row["Participant_ID"],
        "Timestamp": row["Timestamp"],
        "Sensor": "SII",
        "Measurement": "Mean_SI",
        "Observed_Value": row["Mean_SI"],
        "Reason": "Unusually large sound-intensity observation",
        "Review_Status": "Requires investigation"
    })


# Convert records into a DataFrame
quality_review_audit = pd.DataFrame(quality_review_records)

quality_review_audit = quality_review_audit.sort_values(
    ["Sensor", "Participant_ID", "Timestamp"]
).reset_index(drop=True)


# Display audit
print("Total observations flagged:", len(quality_review_audit))

display(quality_review_audit)

print("\nFlags by sensor:")
display(
    quality_review_audit.groupby("Sensor").size().to_frame("Flagged_Records")
)


### Step 36: Consolidated Suspicious Observation Register

A consolidated register of suspicious observations was created using findings from the previous measurement-quality investigations.

#### Observed results

A total of 24 observations were flagged:

| Sensor | Number of flagged observations | Reason |
|---|---:|---|
| EI | 13 | All four AQI measurements equal zero despite valid flags |
| SII | 2 | Unusually large Mean_SI observations |
| SQI | 9 | Negative sleep-efficiency percentages |
| **Total** | **24** | |

#### Interpretation

These observations require further investigation because their values are unusual or inconsistent with the expected measurement meaning.

However, the available dataset documentation does not establish the cause of these observations or provide enough information to correct them reliably.

#### Cleaning decision

- Retain all 24 observations in the cleaned datasets.
- Do not replace suspicious values with zero, NaN, or estimated values.
- Preserve the original validity flags.
- Maintain the consolidated register as a separate audit artifact.
- Distinguish data-quality concerns from unusual but potentially meaningful physiological observations.

The register is intended to support transparency and future quality-aware feature engineering. It does not establish that the flagged observations are definitively erroneous.

### Step 37: Final Cleaning Policy

The cleaning decisions for all eight sensor datasets have been consolidated following the duplicate, missingness, validity-flag, and measurement-range investigations.

| Sensor | Final decision |
|---|---|
| BRI | Preserve missing measurement values. No rows removed. |
| EI | Resolve duplicate timestamps using the documented duplicate policy. Retain suspicious all-zero AQI records. |
| MI | Preserve all records, including non-zero measurements with invalid index flags. |
| PEI | Preserve missing measurement values and unusual physiological observations. |
| PRVI | Resolve duplicate timestamps using the completeness-based policy. Retain records with no available measurements. |
| SHRI | Preserve all records. |
| SII | Resolve duplicate timestamps using the documented duplicate policy. Retain suspicious sound-intensity observations. |
| SQI | Preserve missing values, structurally sparse columns, and suspicious negative sleep-efficiency observations. |

#### General cleaning principles

1. Preserve original measurement values wherever there is insufficient evidence to correct them.
2. Preserve validity flags, except for the documented consolidation of conflicting EI/SII duplicate `data_validity` values, which are represented as missing.
3. Do not perform imputation, interpolation, forward filling, or backfilling.
4. Do not apply scaling or normalization.
5. Do not remove observations solely because they are statistical outliers.
6. Maintain separate audit files for duplicate-resolution decisions and suspicious observations.
7. Keep the original standardized datasets unchanged.

The cleaned datasets will be used as the foundation for subsequent preprocessing and feature engineering.

In [ ]:

# Expected row counts after documented duplicate resolution
expected_rows = {
    "BRI": 9866,
    "EI": 19662,
    "MI": 9908,
    "PEI": 9907,
    "PRVI": 2722,
    "SHRI": 9866,
    "SII": 19662,
    "SQI": 9866
}

summary_rows = []

for sensor, df in all_cleaned_sensors.items():

    measurement_columns = [
        col for col in df.columns
        if col not in [
            "Timestamp", "Participant_ID", "Country", "Sensor",
            "index_validity", "data_validity",
            "data_validity_conflict"
        ]
        and pd.api.types.is_numeric_dtype(df[col])
    ]

    duplicate_count = df.duplicated(
        subset=["Participant_ID", "Timestamp"]
    ).sum()

    missing_measurements = df[measurement_columns].isna().sum().sum()

    summary_rows.append({
        "Sensor": sensor,
        "Actual_Rows": len(df),
        "Expected_Rows": expected_rows[sensor],
        "Row_Count_Match": len(df) == expected_rows[sensor],
        "Participants": df["Participant_ID"].nunique(),
        "Duplicate_Extra_Rows": duplicate_count,
        "Missing_Measurement_Cells": missing_measurements
    })

final_cleaning_summary = pd.DataFrame(summary_rows)

display(final_cleaning_summary)


# Overall validation
print("\nFINAL VALIDATION")

print(
    "All row counts match:",
    final_cleaning_summary["Row_Count_Match"].all()
)

print(
    "No duplicate participant-timestamp keys:",
    (final_cleaning_summary["Duplicate_Extra_Rows"] == 0).all()
)

print(
    "All eight sensors present:",
    set(all_cleaned_sensors.keys()) == set(sensors)
)


### Step 38: Saving Cleaned Sensor Datasets

After validating the final cleaning candidates, all eight cleaned sensor datasets are saved in Parquet format under:

`data/processed/cleaned/`

The cleaned files are saved separately from the standardized source files. Duplicate-resolution audit files are also retained:

- PRVI row-level and duplicate-group audit files.
- EI row-level and duplicate-group audit files.
- SII row-level and duplicate-group audit files.
- The suspicious-observation register, if available in the notebook.

No additional imputation, scaling, interpolation, or outlier removal is performed during saving.

The saved datasets will be read back and compared with their in-memory candidates to verify that the files were written correctly.

In [ ]:

from pathlib import Path

cleaned_path = project_path / "data" / "processed" / "cleaned"
cleaned_path.mkdir(parents=True, exist_ok=True)

# 1. Save all eight cleaned sensor datasets
for sensor, df in all_cleaned_sensors.items():
    file_path = cleaned_path / f"{sensor}.parquet"
    df.to_parquet(file_path, index=False)
    print(f"Saved {sensor}: {len(df)} rows")


# 2. Save PRVI duplicate audit files
prvi_duplicate_audit.to_csv(
    cleaned_path / "PRVI_duplicate_audit.csv",
    index=False
)

prvi_group_audit.to_csv(
    cleaned_path / "PRVI_duplicate_group_summary.csv",
    index=False
)


# 3. Save EI and SII duplicate audit files
for sensor in ["EI", "SII"]:

    ei_sii_row_audits[sensor].to_csv(
        cleaned_path / f"{sensor}_duplicate_audit.csv",
        index=False
    )

    ei_sii_group_audits[sensor].to_csv(
        cleaned_path / f"{sensor}_duplicate_group_summary.csv",
        index=False
    )


# 4. Save the suspicious-observation register, if present
# Find a DataFrame containing the expected register columns.
register_candidates = [
    (name, obj)
    for name, obj in globals().items()
    if isinstance(obj, pd.DataFrame)
    and {"Sensor", "Review_Status"}.issubset(obj.columns)
]

if register_candidates:
    register_name, suspicious_register = register_candidates[-1]

    suspicious_register.to_csv(
        cleaned_path / "suspicious_observation_register.csv",
        index=False
    )

    print(
        f"\nSaved suspicious-observation register "
        f"from variable: {register_name}"
    )
else:
    print("\nNo suspicious-observation register DataFrame found.")


print("\nAll available cleaned datasets and audit files saved.")
print("Output directory:", cleaned_path)


### Step 38C: Read-Back Verification

All cleaned sensor datasets and available audit files have been saved. The saved Parquet files are now loaded back into the notebook and compared with their in-memory cleaned candidates.

The verification checks:

- Whether each saved dataset has the expected number of rows.
- Whether the saved data exactly matches its in-memory candidate.
- Whether duplicate participant-timestamp pairs remain.
- Whether all expected duplicate audit and suspicious-observation files exist.

This confirms that saving did not unintentionally alter the cleaned datasets.

In [ ]:

import pandas as pd
from pathlib import Path

verification_results = []

for sensor, original_df in all_cleaned_sensors.items():

    file_path = cleaned_path / f"{sensor}.parquet"
    saved_df = pd.read_parquet(file_path)

    # Compare saved data with the in-memory candidate
    try:
        pd.testing.assert_frame_equal(
            original_df.reset_index(drop=True),
            saved_df.reset_index(drop=True),
            check_dtype=True,
            check_exact=True
        )
        exact_match = True

    except AssertionError:
        exact_match = False

    duplicate_count = saved_df.duplicated(
        subset=["Participant_ID", "Timestamp"]
    ).sum()

    verification_results.append({
        "Sensor": sensor,
        "Saved_Rows": len(saved_df),
        "In_Memory_Rows": len(original_df),
        "Exact_Match": exact_match,
        "Duplicate_Extra_Rows": duplicate_count
    })


verification_df = pd.DataFrame(verification_results)

display(verification_df)


# Check audit files
expected_audit_files = [
    "PRVI_duplicate_audit.csv",
    "PRVI_duplicate_group_summary.csv",
    "EI_duplicate_audit.csv",
    "EI_duplicate_group_summary.csv",
    "SII_duplicate_audit.csv",
    "SII_duplicate_group_summary.csv",
    "suspicious_observation_register.csv"
]

audit_file_results = []

for filename in expected_audit_files:
    file_path = cleaned_path / filename

    audit_file_results.append({
        "File": filename,
        "Exists": file_path.exists(),
        "Size_Bytes": file_path.stat().st_size if file_path.exists() else None
    })

audit_verification_df = pd.DataFrame(audit_file_results)

display(audit_verification_df)


print("\nFINAL READ-BACK CHECK")

print(
    "All datasets exactly match:",
    verification_df["Exact_Match"].all()
)

print(
    "No duplicate timestamps:",
    (verification_df["Duplicate_Extra_Rows"] == 0).all()
)

print(
    "All audit files exist:",
    audit_verification_df["Exists"].all()
)


## Final Summary: Data Cleaning Completed

The cleaning stage has been completed for all eight sensor datasets.

### 1. Final cleaned dataset sizes

| Sensor | Original standardized rows | Final cleaned rows | Rows removed |
|---|---:|---:|---:|
| BRI | 9,866 | 9,866 | 0 |
| EI | 19,732 | 19,662 | 70 |
| MI | 9,908 | 9,908 | 0 |
| PEI | 9,907 | 9,907 | 0 |
| PRVI | 2,917 | 2,722 | 195 |
| SHRI | 9,866 | 9,866 | 0 |
| SII | 19,732 | 19,662 | 70 |
| SQI | 9,866 | 9,866 | 0 |
| **Total** | **91,794** | **91,459** | **335** |

The 335 removed rows were resolved duplicate records:
- EI: 70
- SII: 70
- PRVI: 195

### 2. Cleaning decisions

- Duplicate timestamps were resolved using documented, sensor-specific policies.
- Missing measurement values were preserved.
- Validity flags were preserved, with conflicting EI/SII duplicate `data_validity` values recorded as missing.
- Suspicious measurements were retained and documented in a separate observation register.
- No imputation, interpolation, scaling, or automatic outlier removal was performed.
- Original standardized datasets were not modified.

### 3. Saved outputs

All eight cleaned Parquet files and seven audit CSV files were saved under:

`data/processed/cleaned/`

### 4. Final verification

The following checks were successfully completed:

- All eight saved datasets exactly match their in-memory candidates.
- All expected row counts match.
- No duplicate participant-timestamp pairs remain.
- All seven expected audit files exist.

**Conclusion:** The eight sensor datasets are now available as cleaned, timestamp-preserving datasets for subsequent preprocessing and feature engineering. Missingness and suspicious observations remain explicitly documented rather than silently altered.